In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Distribution_Network_Optimization")
processed_path = base / "Data" / "Processed"

print("=" * 60)
print("NOTEBOOK 02 PROJECT PATHS")
print("=" * 60)

print()
print("Base path     :", base)
print("Processed path:", processed_path)
print("Processed path exists:", processed_path.exists())

NOTEBOOK 02 PROJECT PATHS

Base path     : /content/drive/MyDrive/Distribution_Network_Optimization
Processed path: /content/drive/MyDrive/Distribution_Network_Optimization/Data/Processed
Processed path exists: True


In [3]:
import pandas as pd
import numpy as np

final_candidate_pool = pd.read_csv(
    processed_path / "final_candidate_pool.csv"
)

customer_demand_regional = pd.read_csv(
    processed_path / "customer_demand_regional.csv"
)

freight_rates = pd.read_csv(
    processed_path / "freight_rates_processed.csv"
)

vmi_check = pd.read_csv(
    processed_path / "vmi_customer_validation.csv"
)

print("=" * 60)
print("NOTEBOOK 02 INPUTS LOADED")
print("=" * 60)

print()
print("Final candidate pool :", final_candidate_pool.shape)
print("Regional demand      :", customer_demand_regional.shape)
print("Freight rates        :", freight_rates.shape)
print("VMI validation       :", vmi_check.shape)

print()
print("Candidate plants :", final_candidate_pool["Plant Code"].nunique())
print("Regional zones   :", customer_demand_regional["city"].nunique())

NOTEBOOK 02 INPUTS LOADED

Final candidate pool : (16, 14)
Regional demand      : (18, 7)
Freight rates        : (1540, 11)
VMI validation       : (14, 3)

Candidate plants : 16
Regional zones   : 18


In [4]:
print("=" * 60)
print("NOTEBOOK 02 INPUT COLUMN CHECK")
print("=" * 60)

print()
print("Candidate pool columns:")
print(final_candidate_pool.columns.tolist())

print()
print("Regional demand columns:")
print(customer_demand_regional.columns.tolist())

print()
print("Freight rate columns:")
print(freight_rates.columns.tolist())

print()
print("VMI validation columns:")
print(vmi_check.columns.tolist())

NOTEBOOK 02 INPUT COLUMN CHECK

Candidate pool columns:
['Plant Code', 'Cost/unit', 'Daily Capacity', 'is_active', 'city', 'latitude', 'longitude', 'screening_status', 'Capacity_wave', 'city_tier', 'footprint_sqft', 'rent_per_sqft_month', 'fixed_cost_annual', 'Cost_per_unit_INR']

Regional demand columns:
['city', 'total_units_day', 'total_weight_day', 'order_count', 'customer_count', 'latitude', 'longitude']

Freight rate columns:
['Carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd', 'minimum cost', 'rate', 'mode_dsc', 'tpt_day_cnt', 'Carrier type']

VMI validation columns:
['Plant Code', 'Customers', 'customer_has_demand']


In [5]:
plant_ports = pd.read_csv(
    processed_path / "plant_ports.csv"
) if (processed_path / "plant_ports.csv").exists() else None

if plant_ports is None:
    raw_path = base / "Data" / "Raw"
    file_path = raw_path / "Supply chain logistics problem.xlsx"
    plant_ports = pd.read_excel(
        file_path,
        sheet_name="PlantPorts"
    )

plant_ports.columns = plant_ports.columns.str.strip()

plant_ports["Plant Code"] = (
    plant_ports["Plant Code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

plant_ports["Port"] = (
    plant_ports["Port"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("=" * 60)
print("PLANT PORT INPUT")
print("=" * 60)

print()
print("Shape :", plant_ports.shape)
print("Unique plants :", plant_ports["Plant Code"].nunique())
print("Unique ports :", plant_ports["Port"].nunique())

print()
print(
    plant_ports
    .sort_values(["Plant Code", "Port"])
    .to_string(index=False)
)

PLANT PORT INPUT

Shape : (22, 2)
Unique plants : 19
Unique ports : 11

Plant Code   Port
   PLANT01 PORT01
   PLANT01 PORT02
   PLANT02 PORT03
   PLANT03 PORT04
   PLANT04 PORT05
   PLANT05 PORT06
   PLANT06 PORT06
   PLANT07 PORT01
   PLANT07 PORT02
   PLANT08 PORT04
   PLANT09 PORT04
   PLANT10 PORT01
   PLANT10 PORT02
   PLANT11 PORT04
   PLANT12 PORT04
   PLANT13 PORT04
   PLANT14 PORT07
   PLANT15 PORT08
   PLANT16 PORT09
   PLANT17 PORT10
   PLANT18 PORT11
   PLANT19 PORT04


In [7]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

CIRCUITY_FACTOR = 1.3

plant_coords = (
    final_candidate_pool
    .set_index("Plant Code")[["latitude", "longitude"]]
    .to_dict("index")
)

zone_coords = (
    customer_demand_regional
    .set_index("city")[["latitude", "longitude"]]
    .to_dict("index")
)

dist_records = []

for plant in final_candidate_pool["Plant Code"]:
    for zone in customer_demand_regional["city"]:
        distance = haversine_km(
            plant_coords[plant]["latitude"],
            plant_coords[plant]["longitude"],
            zone_coords[zone]["latitude"],
            zone_coords[zone]["longitude"]
        ) * CIRCUITY_FACTOR

        dist_records.append({
            "Plant Code": plant,
            "city": zone,
            "distance_km": round(distance, 1)
        })

plant_zone_distance = pd.DataFrame(dist_records)

print("=" * 60)
print("PLANT → REGIONAL ZONE DISTANCE MATRIX")
print("=" * 60)

print()
print("Distance rows :", len(plant_zone_distance))
print("Unique plants :", plant_zone_distance["Plant Code"].nunique())
print("Unique zones  :", plant_zone_distance["city"].nunique())
print("Missing distances :", plant_zone_distance["distance_km"].isna().sum())

print()
print(
    "Distance range : "
    f"{plant_zone_distance['distance_km'].min():.1f} - "
    f"{plant_zone_distance['distance_km'].max():.1f} km"
)

print()
print("Shortest lanes:")
print(
    plant_zone_distance
    .sort_values("distance_km")
    .head(5)
    .to_string(index=False)
)

print()
print("Longest lanes:")
print(
    plant_zone_distance
    .sort_values("distance_km", ascending=False)
    .head(5)
    .to_string(index=False)
)

PLANT → REGIONAL ZONE DISTANCE MATRIX

Distance rows : 288
Unique plants : 16
Unique zones  : 18
Missing distances : 0

Distance range : 0.0 - 3157.7 km

Shortest lanes:
Plant Code      city  distance_km
   PLANT05 Delhi NCR          0.0
   PLANT02    Jaipur          0.0
   PLANT06    Bhopal          0.0
   PLANT03   Kolkata          0.0
   PLANT01      Pune          0.0

Longest lanes:
Plant Code       city  distance_km
   PLANT16   Guwahati       3157.7
   PLANT16 Chandigarh       3007.8
   PLANT04      Kochi       3007.8
   PLANT14   Guwahati       2974.7
   PLANT14 Chandigarh       2850.2


In [9]:
print("=" * 60)
print("CURRENT FREIGHT RATE COLUMNS")
print("=" * 60)

print(freight_rates.columns.tolist())
print()
print("Shape :", freight_rates.shape)
print()
print(freight_rates.head())

CURRENT FREIGHT RATE COLUMNS
['Carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd', 'minimum cost', 'rate', 'mode_dsc', 'tpt_day_cnt', 'Carrier type']

Shape : (1540, 11)

  Carrier orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty svc_cd  \
0  V444_6       PORT08       PORT09         250.0       499.99    DTD   
1  V444_6       PORT08       PORT09          65.0        69.99    DTD   
2  V444_6       PORT08       PORT09          60.0        64.99    DTD   
3  V444_6       PORT08       PORT09          50.0        54.99    DTD   
4  V444_6       PORT08       PORT09          35.0        39.99    DTD   

   minimum cost    rate mode_dsc  tpt_day_cnt Carrier type  
0       43.2272  0.7132   AIR               2  V88888888_0  
1       43.2272  0.7512   AIR               2  V88888888_0  
2       43.2272  0.7892   AIR               2  V88888888_0  
3       43.2272  0.8272   AIR               2  V88888888_0  
4       43.2272  1.0552   AIR               2  V888888

In [12]:
customer_demand = pd.read_csv(
    processed_path / "customer_demand.csv"
)

customer_demand.columns = customer_demand.columns.str.strip()

print("=" * 60)
print("CUSTOMER DEMAND INPUT")
print("=" * 60)

print()
print("Shape :", customer_demand.shape)
print("Columns :", customer_demand.columns.tolist())

print()
print(
    customer_demand[
        ["Customer", "city"]
    ].head(10).to_string(index=False)
)

CUSTOMER DEMAND INPUT

Shape : (46, 7)
Columns : ['Customer', 'order_count', 'total_units_day', 'total_weight_day', 'city', 'latitude', 'longitude']

              Customer       city
V555555555555555555_17  Delhi NCR
V555555555555555555_42     Mumbai
V555555555555555555_45     Bhopal
V555555555555555555_46     Bhopal
   V555555555555555_23 Coimbatore
   V555555555555555_29    Lucknow
   V555555555555555_44   Guwahati
     V55555555555555_8 Chandigarh
     V5555555555555_16 Chandigarh
      V555555555555_31     Nagpur


In [13]:
order_list = pd.read_excel(
    file_path,
    sheet_name="OrderList"
)

order_list.columns = order_list.columns.str.strip()

customer_city_map = (
    customer_demand[["Customer", "city"]]
    .drop_duplicates()
)

order_zone_ports = order_list[
    ["Order ID", "Customer", "Destination Port", "Weight"]
].merge(
    customer_city_map,
    on="Customer",
    how="left"
)

zone_port_summary = (
    order_zone_ports
    .groupby(["city", "Destination Port"])
    .agg(
        order_count=("Order ID", "count"),
        total_weight_kg=("Weight", "sum")
    )
    .reset_index()
    .sort_values(
        ["city", "total_weight_kg"],
        ascending=[True, False]
    )
)

print("=" * 60)
print("REGIONAL ZONE → DESTINATION PORT STRUCTURE")
print("=" * 60)

print()
print("Zones :", zone_port_summary["city"].nunique())
print(
    "Destination ports :",
    zone_port_summary["Destination Port"].nunique()
)

print()
print(zone_port_summary.to_string(index=False))

REGIONAL ZONE → DESTINATION PORT STRUCTURE

Zones : 18
Destination ports : 1

         city Destination Port  order_count  total_weight_kg
    Ahmedabad           PORT09          380      5343.798448
    Bengaluru           PORT09          368      6653.614384
       Bhopal           PORT09          592      9430.857633
   Chandigarh           PORT09         1760     29329.459590
      Chennai           PORT09         1115     14341.743358
   Coimbatore           PORT09          341     16571.403311
    Delhi NCR           PORT09         1504      8622.451359
     Guwahati           PORT09            9       446.572337
    Hyderabad           PORT09           38      1466.215182
       Indore           PORT09          185      2955.324243
       Jaipur           PORT09          220      1554.118818
        Kochi           PORT09          775     18988.482377
      Kolkata           PORT09          521      7298.371246
      Lucknow           PORT09          392      4672.036412
       

In [40]:
DEST_PORT = "PORT09"
USD_TO_INR = 95
PORT_ZONE_RATE = 5

def greedy_tariff_cost(weight_kg, tariff_options):
    if tariff_options.empty:
        return None

    remaining = float(weight_kg)
    total_cost = 0.0

    while remaining > 1e-9:
        max_available = tariff_options["max_wgh_qty"].max()
        shipment_weight = min(remaining, float(max_available))

        applicable = tariff_options[
            (tariff_options["minm_wgh_qty"] <= shipment_weight) &
            (tariff_options["max_wgh_qty"] >= shipment_weight)
        ].copy()

        if applicable.empty:
            return None

        applicable["cost"] = applicable.apply(
            lambda r: max(
                float(r["minimum cost"]),
                shipment_weight * float(r["rate"])
            ),
            axis=1
        )

        row = applicable.sort_values(
            ["cost", "rate", "minimum cost"]
        ).iloc[0]

        total_cost += float(row["cost"])
        remaining -= shipment_weight

    return total_cost


def get_plant_port_cost(plant, weight_kg, require_2day=False):
    ports = plant_ports.loc[
        plant_ports["Plant Code"] == plant,
        "Port"
    ].tolist()

    best_cost = None

    for port in ports:
        options = freight_rates[
            (freight_rates["orig_port_cd"] == port) &
            (freight_rates["dest_port_cd"] == DEST_PORT)
        ].copy()

        if require_2day:
            options = options[
                options["tpt_day_cnt"] <= 2
            ]

        cost = greedy_tariff_cost(
            weight_kg,
            options
        )

        if cost is not None:
            if best_cost is None or cost < best_cost:
                best_cost = cost

    return best_cost


zone_weights = (
    customer_demand_regional
    .set_index("city")["total_weight_day"]
    .to_dict()
)

distance_lookup = (
    plant_zone_distance
    .set_index(["Plant Code", "city"])["distance_km"]
    .to_dict()
)

plants = final_candidate_pool["Plant Code"].tolist()
zones = customer_demand_regional["city"].tolist()

model_a_records = []
model_b_records = []

for plant in plants:
    for zone in zones:

        weight = float(zone_weights[zone])

        distance_km = float(
            distance_lookup[(plant, zone)]
        )

        plant_port_a_usd = get_plant_port_cost(
            plant,
            weight,
            require_2day=False
        )

        plant_port_b_usd = get_plant_port_cost(
            plant,
            weight,
            require_2day=True
        )

        plant_zone_cost_inr = (
            weight / 1000
        ) * distance_km * PORT_ZONE_RATE

        if plant_port_a_usd is not None:
            plant_port_a_inr = (
                plant_port_a_usd * USD_TO_INR
            )

            total_a_inr = (
                plant_port_a_inr +
                plant_zone_cost_inr
            )
        else:
            plant_port_a_inr = np.nan
            total_a_inr = np.nan

        if plant_port_b_usd is not None:
            plant_port_b_inr = (
                plant_port_b_usd * USD_TO_INR
            )

            total_b_inr = (
                plant_port_b_inr +
                plant_zone_cost_inr
            )
        else:
            plant_port_b_inr = np.nan
            total_b_inr = np.nan

        model_a_records.append({
            "Plant Code": plant,
            "city": zone,
            "zone_weight_kg": weight,
            "distance_km": distance_km,
            "plant_port_cost_inr": plant_port_a_inr,
            "plant_zone_cost_inr": plant_zone_cost_inr,
            "total_lane_cost_inr": total_a_inr,
            "is_feasible": plant_port_a_usd is not None
        })

        model_b_records.append({
            "Plant Code": plant,
            "city": zone,
            "zone_weight_kg": weight,
            "distance_km": distance_km,
            "plant_port_cost_inr": plant_port_b_inr,
            "plant_zone_cost_inr": plant_zone_cost_inr,
            "total_lane_cost_inr": total_b_inr,
            "is_feasible": plant_port_b_usd is not None
        })

model_a_lanes = pd.DataFrame(model_a_records)
model_b_lanes = pd.DataFrame(model_b_records)

model_a_lanes.to_csv(
    processed_path / "model_a_lane_costs.csv",
    index=False
)

model_b_lanes.to_csv(
    processed_path / "model_b_lane_costs.csv",
    index=False
)

print("=" * 70)
print("CORRECTED LANE COST MATRICES")
print("=" * 70)

print()
print("Model A lanes :", len(model_a_lanes))
print(
    "Model A infeasible :",
    (~model_a_lanes["is_feasible"]).sum()
)

print()
print("Model B lanes :", len(model_b_lanes))
print(
    "Model B infeasible :",
    (~model_b_lanes["is_feasible"]).sum()
)

print()
print("Model B infeasible plant-zone pairs:")

print(
    model_b_lanes[
        ~model_b_lanes["is_feasible"]
    ][
        ["Plant Code", "city", "zone_weight_kg"]
    ].to_string(index=False)
)

print()
print("=" * 70)
print("PLANT16 — 100 KG SPOT CHECK")
print("=" * 70)

spot_a_usd = get_plant_port_cost(
    "PLANT16",
    100,
    require_2day=False
)

spot_b_usd = get_plant_port_cost(
    "PLANT16",
    100,
    require_2day=True
)

print(
    "Unrestricted : ₹",
    round(spot_a_usd * USD_TO_INR, 2)
)

print(
    "2-day        : ₹",
    round(spot_b_usd * USD_TO_INR, 2)
)

print()
print("Files saved:")
print("model_a_lane_costs.csv")
print("model_b_lane_costs.csv")

CORRECTED LANE COST MATRICES

Model A lanes : 288
Model A infeasible : 0

Model B lanes : 288
Model B infeasible : 18

Model B infeasible plant-zone pairs:
Plant Code          city  zone_weight_kg
   PLANT14     Ahmedabad     5343.798448
   PLANT14     Bengaluru     6653.614384
   PLANT14        Bhopal     9430.857633
   PLANT14    Chandigarh    29329.459590
   PLANT14       Chennai    14341.743358
   PLANT14    Coimbatore    16571.403311
   PLANT14     Delhi NCR     8622.451359
   PLANT14      Guwahati      446.572337
   PLANT14     Hyderabad     1466.215182
   PLANT14        Indore     2955.324243
   PLANT14        Jaipur     1554.118818
   PLANT14         Kochi    18988.482377
   PLANT14       Kolkata     7298.371246
   PLANT14       Lucknow     4672.036412
   PLANT14        Mumbai      297.547486
   PLANT14        Nagpur     3937.761060
   PLANT14          Pune     1341.188818
   PLANT14 Visakhapatnam    49866.654463

PLANT16 — 100 KG SPOT CHECK
Unrestricted : ₹ 1680.44
2-day      

In [15]:
plant16_tariffs = freight_rates[
    (freight_rates["orig_port_cd"] == "PORT09") &
    (freight_rates["dest_port_cd"] == "PORT09")
].copy()

plant16_tariffs = plant16_tariffs[
    [
        "Carrier",
        "orig_port_cd",
        "dest_port_cd",
        "minm_wgh_qty",
        "max_wgh_qty",
        "svc_cd",
        "minimum cost",
        "rate",
        "mode_dsc",
        "tpt_day_cnt",
        "Carrier type"
    ]
].sort_values(
    ["tpt_day_cnt", "minm_wgh_qty", "rate"]
)

print("=" * 60)
print("PLANT16 / PORT09 → PORT09 TARIFFS")
print("=" * 60)

print()
print("Total tariff rows :", len(plant16_tariffs))

print()
print("All tariffs:")
print(
    plant16_tariffs.to_string(index=False)
)

PLANT16 / PORT09 → PORT09 TARIFFS

Total tariff rows : 24

All tariffs:
Carrier orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty svc_cd  minimum cost    rate mode_dsc  tpt_day_cnt Carrier type
 V444_8       PORT09       PORT09           0.0        99.99    DTD       27.3040  0.0588   AIR               0  V88888888_0
 V444_0       PORT09       PORT09           0.0      5000.00    DTP       31.2784 12.2784   GROUND            0  V88888888_0
 V444_0       PORT09       PORT09           0.0      5000.00    DTD       31.2784 13.2784   GROUND            0  V88888888_0
 V444_5       PORT09       PORT09           0.0      5000.00    DTP       31.2784 20.2784   GROUND            0  V88888888_0
 V444_8       PORT09       PORT09         100.0       249.99    DTD       27.3040  0.0588   AIR               0  V88888888_0
 V444_8       PORT09       PORT09         250.0       499.99    DTD       27.3040  0.0588   AIR               0  V88888888_0
 V444_8       PORT09       PORT09         500.0      

In [16]:
print("=" * 60)
print("FREIGHT COST FORMULA RECONCILIATION")
print("=" * 60)

print()
print("Previously validated total freight:")
print("$20,621.30")

print()
print("Current PLANT16 test:")
print("Unrestricted = minimum cost + rate × weight")
print("2-day        = minimum cost + rate × weight")

print()
print("Mentor checkpoint:")
print("Unrestricted = ₹1,680.44 = $17.6888")
print("2-day        = ₹2,430.48 = $25.5840")

FREIGHT COST FORMULA RECONCILIATION

Previously validated total freight:
$20,621.30

Current PLANT16 test:
Unrestricted = minimum cost + rate × weight
2-day        = minimum cost + rate × weight

Mentor checkpoint:
Unrestricted = ₹1,680.44 = $17.6888
2-day        = ₹2,430.48 = $25.5840


In [17]:
print("=" * 60)
print("ORDER-LEVEL FREIGHT FORMULA TEST")
print("=" * 60)

print()
print("We will compare two formulas:")
print("A = minimum cost")
print("B = minimum cost + rate × weight")
print()

print("Do not modify any data or lane matrices yet.")

ORDER-LEVEL FREIGHT FORMULA TEST

We will compare two formulas:
A = minimum cost
B = minimum cost + rate × weight

Do not modify any data or lane matrices yet.


In [18]:
def get_freight_cost_inr(plant_code, weight, require_2day=False):
    options = plant_freight_options[
        plant_freight_options["Plant Code"] == plant_code
    ].copy()

    if require_2day:
        options = options[
            options["meets_2day"]
        ]

    valid = options[
        (valid["minm_wgh_qty"] <= weight) &
        (weight <= valid["max_wgh_qty"])
    ]

    if valid.empty:
        return None

    costs = valid.apply(
        lambda r: max(
            r["minimum cost_INR"],
            weight * r["rate_INR"]
        ),
        axis=1
    )

    return costs.min()

In [20]:
plant_freight_options = (
    plant_ports[
        plant_ports["Plant Code"].isin(
            final_candidate_pool["Plant Code"]
        )
    ]
    .merge(
        freight_rates,
        left_on="Port",
        right_on="orig_port_cd",
        how="left"
    )
)

plant_freight_options["minimum cost_INR"] = (
    plant_freight_options["minimum cost"] * 95
)

plant_freight_options["rate_INR"] = (
    plant_freight_options["rate"] * 95
)

plant_freight_options["meets_2day"] = (
    plant_freight_options["tpt_day_cnt"] <= 2
)

print("=" * 60)
print("PLANT FREIGHT OPTIONS")
print("=" * 60)

print()
print("Rows :", len(plant_freight_options))
print(
    "Plants :",
    plant_freight_options["Plant Code"].nunique()
)

print()
print(
    plant_freight_options[
        [
            "Plant Code",
            "Port",
            "orig_port_cd",
            "dest_port_cd",
            "minm_wgh_qty",
            "max_wgh_qty",
            "minimum cost_INR",
            "rate_INR",
            "tpt_day_cnt",
            "meets_2day"
        ]
    ].head(10).to_string(index=False)
)

PLANT FREIGHT OPTIONS

Rows : 3372
Plants : 16

Plant Code   Port orig_port_cd dest_port_cd  minm_wgh_qty  max_wgh_qty  minimum cost_INR  rate_INR  tpt_day_cnt  meets_2day
   PLANT01 PORT01          NaN          NaN           NaN          NaN               NaN       NaN          NaN       False
   PLANT01 PORT02       PORT02       PORT09         100.0       249.99          1663.982     4.598         14.0       False
   PLANT01 PORT02       PORT02       PORT09           0.0        99.99          1663.982     4.598         14.0       False
   PLANT01 PORT02       PORT02       PORT09         250.0       499.99          1663.982     4.598         14.0       False
   PLANT01 PORT02       PORT02       PORT09        2000.0     99999.99          1663.982     4.598         14.0       False
   PLANT01 PORT02       PORT02       PORT09         500.0      1999.99          1663.982     4.598         14.0       False
   PLANT01 PORT02       PORT02       PORT09        2000.0     99999.99          2214

In [21]:
plant16_unrestricted = get_freight_cost_inr(
    "PLANT16",
    100,
    require_2day=False
)

plant16_2day = get_freight_cost_inr(
    "PLANT16",
    100,
    require_2day=True
)

print("=" * 60)
print("PLANT16 FREIGHT SPOT CHECK")
print("=" * 60)

print()
print("Unrestricted : ₹", round(plant16_unrestricted, 2))
print("2-day        : ₹", round(plant16_2day, 2))

PLANT16 FREIGHT SPOT CHECK

Unrestricted : ₹ 1680.44
2-day        : ₹ 2430.48


In [36]:
model_a_records = []
model_b_records = []

for plant in final_candidate_pool["Plant Code"]:
    for zone in customer_demand_regional["city"]:

        weight = float(
            zone_weights[zone]
        )

        distance_km = float(
            distance_lookup[(plant, zone)]
        )

        plant_port_a_inr = get_freight_cost_inr(
            plant,
            weight,
            require_2day=False
        )

        plant_port_b_inr = get_freight_cost_inr(
            plant,
            weight,
            require_2day=True
        )

        plant_zone_cost_inr = (
            weight / 1000
        ) * distance_km * 5

        if plant_port_a_inr is not None:
            total_a_inr = (
                plant_port_a_inr +
                plant_zone_cost_inr
            )
        else:
            total_a_inr = np.nan

        if plant_port_b_inr is not None:
            total_b_inr = (
                plant_port_b_inr +
                plant_zone_cost_inr
            )
        else:
            total_b_inr = np.nan

        model_a_records.append({
            "Plant Code": plant,
            "city": zone,
            "zone_weight_kg": weight,
            "distance_km": distance_km,
            "plant_port_cost_inr": plant_port_a_inr,
            "plant_zone_cost_inr": plant_zone_cost_inr,
            "total_lane_cost_inr": total_a_inr,
            "is_feasible": plant_port_a_inr is not None
        })

        model_b_records.append({
            "Plant Code": plant,
            "city": zone,
            "zone_weight_kg": weight,
            "distance_km": distance_km,
            "plant_port_cost_inr": plant_port_b_inr,
            "plant_zone_cost_inr": plant_zone_cost_inr,
            "total_lane_cost_inr": total_b_inr,
            "is_feasible": plant_port_b_inr is not None
        })

model_a_lanes = pd.DataFrame(model_a_records)
model_b_lanes = pd.DataFrame(model_b_records)

model_a_lanes.to_csv(
    processed_path / "model_a_lane_costs.csv",
    index=False
)

model_b_lanes.to_csv(
    processed_path / "model_b_lane_costs.csv",
    index=False
)

print("=" * 60)
print("CORRECTED LANE COST MATRICES")
print("=" * 60)

print()
print("Model A lanes :", len(model_a_lanes))
print("Model A infeasible :", (~model_a_lanes["is_feasible"]).sum())

print()
print("Model B lanes :", len(model_b_lanes))
print("Model B infeasible :", (~model_b_lanes["is_feasible"]).sum())

print()
print("Model B infeasible plant-zone pairs:")

print(
    model_b_lanes[
        ~model_b_lanes["is_feasible"]
    ][
        ["Plant Code", "city", "zone_weight_kg"]
    ].to_string(index=False)
)

print()
print("PLANT16 spot-check:")
print(
    "Unrestricted : ₹",
    round(
        get_freight_cost_inr(
            "PLANT16",
            100,
            False
        ),
        2
    )
)

print(
    "2-day        : ₹",
    round(
        get_freight_cost_inr(
            "PLANT16",
            100,
            True
        ),
        2
    )
)

CORRECTED LANE COST MATRICES

Model A lanes : 288
Model A infeasible : 0

Model B lanes : 288
Model B infeasible : 18

Model B infeasible plant-zone pairs:
Plant Code          city  zone_weight_kg
   PLANT14     Ahmedabad     5343.798448
   PLANT14     Bengaluru     6653.614384
   PLANT14        Bhopal     9430.857633
   PLANT14    Chandigarh    29329.459590
   PLANT14       Chennai    14341.743358
   PLANT14    Coimbatore    16571.403311
   PLANT14     Delhi NCR     8622.451359
   PLANT14      Guwahati      446.572337
   PLANT14     Hyderabad     1466.215182
   PLANT14        Indore     2955.324243
   PLANT14        Jaipur     1554.118818
   PLANT14         Kochi    18988.482377
   PLANT14       Kolkata     7298.371246
   PLANT14       Lucknow     4672.036412
   PLANT14        Mumbai      297.547486
   PLANT14        Nagpur     3937.761060
   PLANT14          Pune     1341.188818
   PLANT14 Visakhapatnam    49866.654463

PLANT16 spot-check:
Unrestricted : ₹ 1680.44
2-day        : ₹ 24

In [37]:
print("=" * 80)
print("LANE COST FORMULA DIAGNOSTIC")
print("=" * 80)

print()
print("Plant → port total, wave:")
print(round(model_a_lanes["plant_port_cost_inr"].sum(), 2))

print("Plant → zone total, wave:")
print(round(model_a_lanes["plant_zone_cost_inr"].sum(), 2))

print("Total lane cost, wave:")
print(round(model_a_lanes["total_lane_cost_inr"].sum(), 2))

print()
print("Plant → port total, annual:")
print(round(model_a_lanes["plant_port_cost_inr"].sum() * 50, 2))

print("Plant → zone total, annual:")
print(round(model_a_lanes["plant_zone_cost_inr"].sum() * 50, 2))

print("Total lane cost, annual:")
print(round(model_a_lanes["total_lane_cost_inr"].sum() * 50, 2))

LANE COST FORMULA DIAGNOSTIC

Plant → port total, wave:
17657262.64
Plant → zone total, wave:
20370997.78
Total lane cost, wave:
38028260.42

Plant → port total, annual:
882863132.11
Plant → zone total, annual:
1018549889.04
Total lane cost, annual:
1901413021.15


In [23]:
plant14_options = plant_freight_options[
    plant_freight_options["Plant Code"] == "PLANT14"
].copy()

print("=" * 60)
print("PLANT14 FREIGHT OPTION VALIDATION")
print("=" * 60)

print()
print("Total freight options for PLANT14 :", len(plant14_options))

print()
print(
    plant14_options[
        [
            "Port",
            "Carrier",
            "mode_dsc",
            "svc_cd",
            "tpt_day_cnt",
            "minm_wgh_qty",
            "max_wgh_qty"
        ]
    ]
    .sort_values(
        ["tpt_day_cnt", "minm_wgh_qty"]
    )
    .to_string(index=False)
)

print()

print(
    "Minimum tpt_day_cnt available :",
    plant14_options["tpt_day_cnt"].min()
)

print()

print("Transit-day distribution:")

print(
    plant14_options["tpt_day_cnt"]
    .value_counts()
    .sort_index()
)

PLANT14 FREIGHT OPTION VALIDATION

Total freight options for PLANT14 : 20

  Port Carrier mode_dsc svc_cd  tpt_day_cnt  minm_wgh_qty  max_wgh_qty
PORT07  V444_2   AIR       DTD          3.0           0.0        99.99
PORT07  V444_2   AIR       DTD          3.0         100.0       249.99
PORT07  V444_2   AIR       DTD          3.0         250.0       499.99
PORT07  V444_2   AIR       DTD          3.0         500.0      1999.99
PORT07  V444_2   AIR       DTD          3.0        2000.0     99999.99
PORT07  V444_2   AIR       DTD          4.0           0.0        99.99
PORT07  V444_2   AIR       DTD          4.0         100.0       249.99
PORT07  V444_2   AIR       DTD          4.0         250.0       499.99
PORT07  V444_2   AIR       DTD          4.0         500.0      1999.99
PORT07  V444_2   AIR       DTD          4.0        2000.0     99999.99
PORT07  V444_2   AIR       DTP          5.0           0.0        99.99
PORT07  V444_2   AIR       DTP          5.0         100.0       249.99
PO

In [24]:
model_a_check = pd.read_csv(
    processed_path / "model_a_lane_costs.csv"
)

model_b_check = pd.read_csv(
    processed_path / "model_b_lane_costs.csv"
)

print("=" * 60)
print("LANE COST ARTIFACT STRUCTURE")
print("=" * 60)

print()
print("Model A shape :", model_a_check.shape)
print("Model A columns :")
print(model_a_check.columns.tolist())

print()
print("Model A sample:")
print(
    model_a_check.head(3).to_string(index=False)
)

print()
print("Model B shape :", model_b_check.shape)
print("Model B columns :")
print(model_b_check.columns.tolist())

print()
print("Model B sample:")
print(
    model_b_check.head(3).to_string(index=False)
)

print()
print("=" * 60)
print("OPTIMIZATION INPUTS IN MEMORY")
print("=" * 60)

print()
print("Fixed cost column:")
print(
    final_candidate_pool[
        ["Plant Code", "fixed_cost_annual"]
    ].head().to_string(index=False)
)

print()
print("Capacity column:")
print(
    final_candidate_pool[
        ["Plant Code", "Capacity_wave"]
    ].head().to_string(index=False)
)

print()
print("Demand orders:")
print(
    customer_demand_regional[
        ["city", "order_count"]
    ].to_string(index=False)
)

print()
print("Active plants:")
print(
    sorted(
        final_candidate_pool.loc[
            final_candidate_pool["is_active"] == 1,
            "Plant Code"
        ].tolist()
    )
)

LANE COST ARTIFACT STRUCTURE

Model A shape : (288, 8)
Model A columns :
['Plant Code', 'city', 'zone_weight_kg', 'distance_km', 'plant_port_cost_inr', 'plant_zone_cost_inr', 'total_lane_cost_inr', 'is_feasible']

Model A sample:
Plant Code      city  zone_weight_kg  distance_km  plant_port_cost_inr  plant_zone_cost_inr  total_lane_cost_inr  is_feasible
   PLANT15 Ahmedabad     5343.798448        572.0         51578.342624         15283.263562         66861.606186         True
   PLANT15 Bengaluru     6653.614384       1098.9         64220.686035         36558.284233        100778.970269         True
   PLANT15    Bhopal     9430.857633        859.8         91026.637874         40543.256965        131569.894839         True

Model B shape : (288, 8)
Model B columns :
['Plant Code', 'city', 'zone_weight_kg', 'distance_km', 'plant_port_cost_inr', 'plant_zone_cost_inr', 'total_lane_cost_inr', 'is_feasible']

Model B sample:
Plant Code      city  zone_weight_kg  distance_km  plant_port_cos

In [43]:
import pulp
import pandas as pd
import numpy as np

FIXED_COST_WAVE_FACTOR = 5 / 250
ANNUALIZATION_FACTOR = 50
PLANNING_DAYS = 5

plants = final_candidate_pool["Plant Code"].tolist()
zones = customer_demand_regional["city"].tolist()

active_plants = sorted(
    final_candidate_pool.loc[
        final_candidate_pool["is_active"] == 1,
        "Plant Code"
    ].tolist()
)

zone_orders = (
    customer_demand_regional
    .set_index("city")["order_count"]
    .astype(float)
    .to_dict()
)

zone_units_day = (
    customer_demand_regional
    .set_index("city")["total_units_day"]
    .astype(float)
    .to_dict()
)

capacity_wave = (
    final_candidate_pool
    .set_index("Plant Code")["Capacity_wave"]
    .astype(float)
    .to_dict()
)

fixed_cost_wave = (
    final_candidate_pool
    .set_index("Plant Code")["fixed_cost_annual"]
    .astype(float)
    .mul(FIXED_COST_WAVE_FACTOR)
    .to_dict()
)

variable_cost_per_unit = (
    final_candidate_pool
    .set_index("Plant Code")["Cost_per_unit_INR"]
    .astype(float)
    .to_dict()
)

def solve_network(lane_df, model_name, baseline):

    lane_lookup = {}

    for _, row in lane_df.iterrows():

        plant = row["Plant Code"]
        zone = row["city"]

        if not bool(row["is_feasible"]):
            continue

        lane_lookup[(plant, zone)] = float(
            row["total_lane_cost_inr"]
        )

    problem = pulp.LpProblem(
        f"{model_name}_{'Baseline' if baseline else 'Optimized'}",
        pulp.LpMinimize
    )

    y = {
        i: pulp.LpVariable(
            f"y_{i}",
            cat="Binary"
        )
        for i in plants
    }

    z = {
        (i, j): pulp.LpVariable(
            f"z_{i}_{j}",
            cat="Binary"
        )
        for (i, j) in lane_lookup
    }

    objective_terms = []

    for i in plants:
        objective_terms.append(
            fixed_cost_wave[i] * y[i]
        )

    for (i, j), var in z.items():

        variable_cost_wave = (
            variable_cost_per_unit[i]
            * zone_units_day[j]
            * PLANNING_DAYS
        )

        freight_cost_wave = lane_lookup[(i, j)]

        objective_terms.append(
            (
                variable_cost_wave
                + freight_cost_wave
            ) * var
        )

    problem += pulp.lpSum(objective_terms)

    for j in zones:

        feasible_plants = [
            i
            for i in plants
            if (i, j) in z
        ]

        problem += (
            pulp.lpSum(
                z[(i, j)]
                for i in feasible_plants
            ) == 1,
            f"single_source_{j}"
        )

    for i in plants:

        assigned_orders = [
            zone_orders[j] * z[(i, j)]
            for j in zones
            if (i, j) in z
        ]

        problem += (
            pulp.lpSum(assigned_orders)
            <= capacity_wave[i] * y[i],
            f"capacity_{i}"
        )

    for (i, j), var in z.items():

        problem += (
            var <= y[i],
            f"open_link_{i}_{j}"
        )

    if baseline:

        for i in plants:

            if i in active_plants:
                problem += (
                    y[i] == 1,
                    f"baseline_open_{i}"
                )
            else:
                problem += (
                    y[i] == 0,
                    f"baseline_closed_{i}"
                )

    problem.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    status = pulp.LpStatus[problem.status]

    if status != "Optimal":

        return {
            "model": model_name,
            "scenario": "Baseline" if baseline else "Optimized",
            "status": status,
            "wave_objective": np.nan,
            "annual_objective": np.nan,
            "open_plants": [],
            "assignments": pd.DataFrame()
        }

    wave_objective = pulp.value(problem.objective)

    annual_objective = (
        wave_objective * ANNUALIZATION_FACTOR
    )

    open_plants = [
        i
        for i in plants
        if pulp.value(y[i]) > 0.5
    ]

    assignment_records = []

    for (i, j), var in z.items():

        if pulp.value(var) > 0.5:

            assignment_records.append({
                "Plant Code": i,
                "city": j,
                "demand_orders_wave": zone_orders[j],
                "demand_units_wave":
                    zone_units_day[j] * PLANNING_DAYS,
                "demand_units_day":
                    zone_units_day[j],
                "variable_cost_per_unit":
                    variable_cost_per_unit[i],
                "variable_cost_wave":
                    variable_cost_per_unit[i]
                    * zone_units_day[j]
                    * PLANNING_DAYS,
                "freight_cost_wave":
                    lane_lookup[(i, j)]
            })

    assignments = pd.DataFrame(
        assignment_records
    )

    return {
        "model": model_name,
        "scenario": "Baseline" if baseline else "Optimized",
        "status": status,
        "wave_objective": wave_objective,
        "annual_objective": annual_objective,
        "open_plants": open_plants,
        "assignments": assignments
    }

results = []

results.append(
    solve_network(
        model_a_lanes,
        "Model_A",
        True
    )
)

results.append(
    solve_network(
        model_a_lanes,
        "Model_A",
        False
    )
)

results.append(
    solve_network(
        model_b_lanes,
        "Model_B",
        True
    )
)

results.append(
    solve_network(
        model_b_lanes,
        "Model_B",
        False
    )
)

print("=" * 70)
print("FOUR-SCENARIO PULP OPTIMIZATION")
print("=" * 70)

for result in results:

    print()

    print(
        result["model"],
        "|",
        result["scenario"]
    )

    print(
        "Status              :",
        result["status"]
    )

    print(
        "Wave objective      : ₹",
        round(
            result["wave_objective"],
            2
        )
    )

    print(
        "Annualized ×50      : ₹",
        round(
            result["annual_objective"],
            2
        )
    )

    print(
        "Open plants         :",
        result["open_plants"]
    )

print()
print("=" * 70)
print("MODEL A BASELINE COMPONENTS")
print("=" * 70)

baseline = results[0]

baseline_assignments = baseline["assignments"].copy()

fixed_wave = sum(
    fixed_cost_wave[p]
    for p in baseline["open_plants"]
)

variable_wave = (
    baseline_assignments["variable_cost_wave"].sum()
)

freight_wave = (
    baseline_assignments["freight_cost_wave"].sum()
)

total_wave = (
    fixed_wave
    + variable_wave
    + freight_wave
)

print(
    "Fixed cost — wave     : ₹",
    round(fixed_wave, 2)
)

print(
    "Variable cost — wave  : ₹",
    round(variable_wave, 2)
)

print(
    "Freight cost — wave   : ₹",
    round(freight_wave, 2)
)

print(
    "Total — wave          : ₹",
    round(total_wave, 2)
)

print()
print(
    "Fixed cost — annual   : ₹",
    round(fixed_wave * 50, 2)
)

print(
    "Variable cost — annual: ₹",
    round(variable_wave * 50, 2)
)

print(
    "Freight cost — annual : ₹",
    round(freight_wave * 50, 2)
)

print(
    "Total — annual        : ₹",
    round(total_wave * 50, 2)
)

print()
print("=" * 70)
print("MODEL SAVINGS")
print("=" * 70)

for model_name, base_idx, opt_idx in [
    ("MODEL A", 0, 1),
    ("MODEL B", 2, 3)
]:

    base = results[base_idx]["annual_objective"]
    opt = results[opt_idx]["annual_objective"]

    savings = base - opt
    savings_pct = (
        savings / base * 100
    )

    print()
    print(model_name)
    print(
        "Baseline annual cost  : ₹",
        round(base, 2)
    )
    print(
        "Optimized annual cost : ₹",
        round(opt, 2)
    )
    print(
        "Annual savings        : ₹",
        round(savings, 2)
    )
    print(
        "Savings %             :",
        round(savings_pct, 4),
        "%"
    )

print()
print("=" * 70)
print("BASELINE ASSIGNMENTS")
print("=" * 70)

print(
    baseline_assignments[
        [
            "Plant Code",
            "city",
            "demand_orders_wave",
            "demand_units_day",
            "variable_cost_per_unit",
            "variable_cost_wave",
            "freight_cost_wave"
        ]
    ]
    .sort_values(
        ["Plant Code", "city"]
    )
    .to_string(index=False)
)

print()
print("=" * 70)
print("ASSIGNMENT FILES")
print("=" * 70)

baseline_assignments.to_csv(
    processed_path / "model_a_baseline_assignments.csv",
    index=False
)

results[1]["assignments"].to_csv(
    processed_path / "model_a_optimized_assignments.csv",
    index=False
)

results[2]["assignments"].to_csv(
    processed_path / "model_b_baseline_assignments.csv",
    index=False
)

results[3]["assignments"].to_csv(
    processed_path / "model_b_optimized_assignments.csv",
    index=False
)

print("model_a_baseline_assignments.csv")
print("model_a_optimized_assignments.csv")
print("model_b_baseline_assignments.csv")
print("model_b_optimized_assignments.csv")

FOUR-SCENARIO PULP OPTIMIZATION

Model_A | Baseline
Status              : Optimal
Wave objective      : ₹ 6268596257.9
Annualized ×50      : ₹ 313429812894.86
Open plants         : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

Model_A | Optimized
Status              : Optimal
Wave objective      : ₹ 5841004341.18
Annualized ×50      : ₹ 292050217058.8
Open plants         : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

Model_B | Baseline
Status              : Optimal
Wave objective      : ₹ 6268617377.52
Annualized ×50      : ₹ 313430868875.89
Open plants         : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

Model_B | Optimized
Status              : Optimal
Wave objective      : ₹ 5841132757.18
Annualized ×50      : ₹ 292056637858.97
Open plants         : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

MODEL A BASELINE COMPONENTS
Fixed cost — wave     : ₹ 842856.0
Variable cost 

In [44]:
print("=" * 80)
print("FINAL NETWORK OPTIMIZATION RESULTS")
print("=" * 80)

for r in results:
    if r["status"] != "Optimal":
        continue

    print()
    print(r["model"], "|", r["scenario"])
    print("Annual cost       : ₹", f'{r["annual_objective"]:,.2f}')
    print("Open plants       :", r["open_plants"])
    print("Number of plants  :", len(r["open_plants"]))
    print("Number of zones   :", len(r["assignments"]))

print()
print("=" * 80)
print("MODEL A KPI")
print("=" * 80)

a_base = results[0]["annual_objective"]
a_opt = results[1]["annual_objective"]

a_savings = a_base - a_opt
a_savings_pct = a_savings / a_base * 100

print("Baseline cost     : ₹", f"{a_base:,.2f}")
print("Optimized cost    : ₹", f"{a_opt:,.2f}")
print("Annual savings    : ₹", f"{a_savings:,.2f}")
print("Savings %         :", f"{a_savings_pct:.4f}%")

print()
print("=" * 80)
print("MODEL B KPI")
print("=" * 80)

b_base = results[2]["annual_objective"]
b_opt = results[3]["annual_objective"]

b_savings = b_base - b_opt
b_savings_pct = b_savings / b_base * 100

print("Baseline cost     : ₹", f"{b_base:,.2f}")
print("Optimized cost    : ₹", f"{b_opt:,.2f}")
print("Annual savings    : ₹", f"{b_savings:,.2f}")
print("Savings %         :", f"{b_savings_pct:.4f}%")

print()
print("=" * 80)
print("MODEL A vs MODEL B")
print("=" * 80)

print("Baseline difference : ₹", f"{b_base - a_base:,.2f}")
print("Optimized difference: ₹", f"{b_opt - a_opt:,.2f}")

FINAL NETWORK OPTIMIZATION RESULTS

Model_A | Baseline
Annual cost       : ₹ 313,429,812,894.86
Open plants       : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']
Number of plants  : 7
Number of zones   : 18

Model_A | Optimized
Annual cost       : ₹ 292,050,217,058.80
Open plants       : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']
Number of plants  : 6
Number of zones   : 18

Model_B | Baseline
Annual cost       : ₹ 313,430,868,875.89
Open plants       : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']
Number of plants  : 7
Number of zones   : 18

Model_B | Optimized
Annual cost       : ₹ 292,056,637,858.97
Open plants       : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']
Number of plants  : 6
Number of zones   : 18

MODEL A KPI
Baseline cost     : ₹ 313,429,812,894.86
Optimized cost    : ₹ 292,050,217,058.80
Annual savings    : ₹ 21,379,595,836.05
Savings %         : 6.8212%

MODEL

In [35]:
baseline_assignments = results[0]["assignments"].copy()

print("=" * 90)
print("MODEL A LANE-COST STRUCTURE CHECK")
print("=" * 90)

print()
print("Model A lane-cost columns:")
print(model_a_lanes.columns.tolist())

print()
print("Model A lane-cost summary:")
print(
    model_a_lanes[
        [
            "zone_weight_kg",
            "distance_km",
            "plant_port_cost_inr",
            "plant_zone_cost_inr",
            "total_lane_cost_inr"
        ]
    ].describe().to_string()
)

print()
print("=" * 90)
print("BASELINE ASSIGNMENT LANE BREAKDOWN")
print("=" * 90)

baseline_lane_details = model_a_lanes.merge(
    baseline_assignments[["Plant Code", "city"]],
    on=["Plant Code", "city"],
    how="inner"
)

print(
    baseline_lane_details[
        [
            "Plant Code",
            "city",
            "zone_weight_kg",
            "distance_km",
            "plant_port_cost_inr",
            "plant_zone_cost_inr",
            "total_lane_cost_inr"
        ]
    ]
    .sort_values(["Plant Code", "city"])
    .to_string(index=False)
)

print()
print("=" * 90)
print("LAST-MILE COST CHECK")
print("=" * 90)

print(
    "Rows with zero plant→zone cost:",
    (
        baseline_lane_details["plant_zone_cost_inr"] == 0
    ).sum()
)

print(
    "Rows with positive plant→zone cost:",
    (
        baseline_lane_details["plant_zone_cost_inr"] > 0
    ).sum()
)

print(
    "Total plant→port wave cost : ₹",
    round(
        baseline_lane_details["plant_port_cost_inr"].sum(),
        2
    )
)

print(
    "Total plant→zone wave cost : ₹",
    round(
        baseline_lane_details["plant_zone_cost_inr"].sum(),
        2
    )
)

print(
    "Total freight wave         : ₹",
    round(
        baseline_lane_details["total_lane_cost_inr"].sum(),
        2
    )
)

print()
print("=" * 90)
print("MODEL A vs MODEL B BASELINE ASSIGNMENTS")
print("=" * 90)

baseline_a = (
    results[0]["assignments"]
    .set_index("city")["Plant Code"]
    .to_dict()
)

baseline_b = (
    results[2]["assignments"]
    .set_index("city")["Plant Code"]
    .to_dict()
)

assignment_diffs = {
    zone: (
        baseline_a.get(zone),
        baseline_b.get(zone)
    )
    for zone in zones
    if baseline_a.get(zone) != baseline_b.get(zone)
}

print(
    "Zones with different assignments:",
    len(assignment_diffs)
)

print(assignment_diffs)

MODEL A LANE-COST STRUCTURE CHECK

Model A lane-cost columns:
['Plant Code', 'city', 'zone_weight_kg', 'distance_km', 'plant_port_cost_inr', 'plant_zone_cost_inr', 'total_lane_cost_inr', 'is_feasible']

Model A lane-cost summary:
       zone_weight_kg  distance_km  plant_port_cost_inr  plant_zone_cost_inr  total_lane_cost_inr
count      288.000000   288.000000           288.000000           288.000000         2.880000e+02
mean     10173.200029  1318.004861         61309.939730         70732.631184         1.320426e+05
std      12193.488415   731.135368         87073.774081         99040.253210         1.740095e+05
min        297.547486     0.000000          1198.521274             0.000000         1.649008e+03
25%       1554.118818   766.350000         11659.398361          7738.915433         2.064669e+04
50%       5998.706416  1289.200000         30593.318938         26888.423378         6.685217e+04
75%      14341.743358  1822.925000         73179.817739         83934.736737        

In [31]:
import pulp
import pandas as pd
import numpy as np

PLANNING_DAYS = 5
ANNUAL_DAYS = 250

FIXED_COST_WAVE_FACTOR = PLANNING_DAYS / ANNUAL_DAYS
ANNUALIZATION_FACTOR = ANNUAL_DAYS / PLANNING_DAYS

EXPECTED_BASELINE = 313_954_719_722

plants = final_candidate_pool["Plant Code"].tolist()
zones = customer_demand_regional["city"].tolist()

active_plants = sorted(
    final_candidate_pool.loc[
        final_candidate_pool["is_active"] == 1,
        "Plant Code"
    ].tolist()
)

zone_orders = (
    customer_demand_regional
    .set_index("city")["order_count"]
    .astype(float)
    .to_dict()
)

zone_units_day = (
    customer_demand_regional
    .set_index("city")["total_units_day"]
    .astype(float)
    .to_dict()
)

capacity_wave = (
    final_candidate_pool
    .set_index("Plant Code")["Capacity_wave"]
    .astype(float)
    .to_dict()
)

fixed_cost_wave = (
    final_candidate_pool
    .set_index("Plant Code")["fixed_cost_annual"]
    .astype(float)
    .mul(FIXED_COST_WAVE_FACTOR)
    .to_dict()
)

variable_cost_per_unit = (
    final_candidate_pool
    .set_index("Plant Code")["Cost_per_unit_INR"]
    .astype(float)
    .to_dict()
)


def build_lane_lookup(lane_df):

    lookup = {}

    for _, row in lane_df.iterrows():

        if not bool(row["is_feasible"]):
            continue

        plant = row["Plant Code"]
        zone = row["city"]

        lookup[(plant, zone)] = float(
            row["total_lane_cost_inr"]
        )

    return lookup


def solve_network(lane_df, model_name, baseline):

    lane_lookup = build_lane_lookup(lane_df)

    problem = pulp.LpProblem(
        f"{model_name}_{'Baseline' if baseline else 'Optimized'}",
        pulp.LpMinimize
    )

    y = {
        plant: pulp.LpVariable(
            f"y_{plant}",
            cat="Binary"
        )
        for plant in plants
    }

    z = {
        (plant, zone): pulp.LpVariable(
            f"z_{plant}_{zone}",
            cat="Binary"
        )
        for (plant, zone) in lane_lookup
    }

    objective_terms = []

    for plant in plants:
        objective_terms.append(
            fixed_cost_wave[plant] * y[plant]
        )

    for (plant, zone), variable in z.items():

        variable_cost_wave = (
            variable_cost_per_unit[plant]
            * zone_units_day[zone]
            * PLANNING_DAYS
        )

        freight_cost_wave = lane_lookup[
            (plant, zone)
        ]

        objective_terms.append(
            (
                variable_cost_wave
                + freight_cost_wave
            ) * variable
        )

    problem += pulp.lpSum(objective_terms)

    for zone in zones:

        feasible_plants = [
            plant
            for plant in plants
            if (plant, zone) in z
        ]

        problem += (
            pulp.lpSum(
                z[(plant, zone)]
                for plant in feasible_plants
            ) == 1
        )

    for plant in plants:

        assigned_orders = [
            zone_orders[zone] * z[(plant, zone)]
            for zone in zones
            if (plant, zone) in z
        ]

        problem += (
            pulp.lpSum(assigned_orders)
            <= capacity_wave[plant] * y[plant]
        )

    for (plant, zone), variable in z.items():

        problem += (
            variable <= y[plant]
        )

    if baseline:

        for plant in plants:

            if plant in active_plants:
                problem += (
                    y[plant] == 1
                )
            else:
                problem += (
                    y[plant] == 0
                )

    problem.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    status = pulp.LpStatus[problem.status]

    if status != "Optimal":

        return {
            "model": model_name,
            "scenario": "Baseline" if baseline else "Optimized",
            "status": status,
            "wave_objective": np.nan,
            "annual_objective": np.nan,
            "open_plants": [],
            "assignments": pd.DataFrame()
        }

    wave_objective = float(
        pulp.value(problem.objective)
    )

    annual_objective = (
        wave_objective * ANNUALIZATION_FACTOR
    )

    open_plants = [
        plant
        for plant in plants
        if pulp.value(y[plant]) > 0.5
    ]

    assignment_records = []

    for (plant, zone), variable in z.items():

        if pulp.value(variable) > 0.5:

            assignment_records.append({
                "Plant Code": plant,
                "city": zone,
                "demand_orders_wave": zone_orders[zone],
                "demand_units_day": zone_units_day[zone],
                "demand_units_wave":
                    zone_units_day[zone] * PLANNING_DAYS,
                "variable_cost_per_unit":
                    variable_cost_per_unit[plant],
                "variable_cost_wave":
                    variable_cost_per_unit[plant]
                    * zone_units_day[zone]
                    * PLANNING_DAYS,
                "freight_cost_wave":
                    lane_lookup[(plant, zone)]
            })

    assignments = pd.DataFrame(
        assignment_records
    )

    return {
        "model": model_name,
        "scenario": "Baseline" if baseline else "Optimized",
        "status": status,
        "wave_objective": wave_objective,
        "annual_objective": annual_objective,
        "open_plants": open_plants,
        "assignments": assignments
    }


results = []

results.append(
    solve_network(
        model_a_lanes,
        "Model_A",
        True
    )
)

results.append(
    solve_network(
        model_a_lanes,
        "Model_A",
        False
    )
)

results.append(
    solve_network(
        model_b_lanes,
        "Model_B",
        True
    )
)

results.append(
    solve_network(
        model_b_lanes,
        "Model_B",
        False
    )
)


print("=" * 70)
print("FOUR-SCENARIO PULP OPTIMIZATION")
print("=" * 70)

for result in results:

    print()
    print(
        result["model"],
        "|",
        result["scenario"]
    )

    print(
        "Status              :",
        result["status"]
    )

    print(
        "Wave objective      : ₹",
        round(result["wave_objective"], 2)
    )

    print(
        "Annualized ×50      : ₹",
        round(result["annual_objective"], 2)
    )

    print(
        "Open plants         :",
        result["open_plants"]
    )


model_a_baseline = results[0]

calculated = model_a_baseline["annual_objective"]

difference = calculated - EXPECTED_BASELINE

print()
print("=" * 70)
print("MODEL A BASELINE VALIDATION")
print("=" * 70)

print(
    "Calculated annualized : ₹",
    round(calculated, 2)
)

print(
    "Previously validated  : ₹",
    EXPECTED_BASELINE
)

print(
    "Difference            : ₹",
    round(difference, 2)
)

print(
    "Exact reproduction    :",
    abs(difference) < 0.01
)


assignments = model_a_baseline["assignments"].copy()

fixed_wave = sum(
    fixed_cost_wave[plant]
    for plant in model_a_baseline["open_plants"]
)

variable_wave = assignments[
    "variable_cost_wave"
].sum()

freight_wave = assignments[
    "freight_cost_wave"
].sum()

reconstructed_wave = (
    fixed_wave
    + variable_wave
    + freight_wave
)

print()
print("=" * 70)
print("MODEL A BASELINE COST DECOMPOSITION")
print("=" * 70)

print(
    "Fixed cost — wave       : ₹",
    round(fixed_wave, 2)
)

print(
    "Variable cost — wave    : ₹",
    round(variable_wave, 2)
)

print(
    "Freight cost — wave     : ₹",
    round(freight_wave, 2)
)

print(
    "Total — wave            : ₹",
    round(reconstructed_wave, 2)
)

print(
    "PuLP objective — wave   : ₹",
    round(
        model_a_baseline["wave_objective"],
        2
    )
)

print(
    "Reconciliation diff     : ₹",
    round(
        reconstructed_wave
        - model_a_baseline["wave_objective"],
        2
    )
)

print()
print(
    "Fixed cost — annual     : ₹",
    round(fixed_wave * 50, 2)
)

print(
    "Variable cost — annual  : ₹",
    round(variable_wave * 50, 2)
)

print(
    "Freight cost — annual   : ₹",
    round(freight_wave * 50, 2)
)

print(
    "Total — annual          : ₹",
    round(reconstructed_wave * 50, 2)
)


def calculate_savings(
    baseline_result,
    optimized_result
):

    baseline_cost = baseline_result[
        "annual_objective"
    ]

    optimized_cost = optimized_result[
        "annual_objective"
    ]

    savings = (
        baseline_cost
        - optimized_cost
    )

    savings_pct = (
        savings
        / baseline_cost
        * 100
    )

    return savings, savings_pct


model_a_optimized = results[1]
model_b_baseline = results[2]
model_b_optimized = results[3]

a_savings, a_savings_pct = calculate_savings(
    model_a_baseline,
    model_a_optimized
)

b_savings, b_savings_pct = calculate_savings(
    model_b_baseline,
    model_b_optimized
)

print()
print("=" * 70)
print("MODEL SAVINGS")
print("=" * 70)

print()
print("MODEL A")

print(
    "Baseline annual cost  : ₹",
    round(
        model_a_baseline["annual_objective"],
        2
    )
)

print(
    "Optimized annual cost : ₹",
    round(
        model_a_optimized["annual_objective"],
        2
    )
)

print(
    "Annual savings        : ₹",
    round(a_savings, 2)
)

print(
    "Savings %             :",
    round(a_savings_pct, 4),
    "%"
)

print()
print("MODEL B")

print(
    "Baseline annual cost  : ₹",
    round(
        model_b_baseline["annual_objective"],
        2
    )
)

print(
    "Optimized annual cost : ₹",
    round(
        model_b_optimized["annual_objective"],
        2
    )
)

print(
    "Annual savings        : ₹",
    round(b_savings, 2)
)

print(
    "Savings %             :",
    round(b_savings_pct, 4),
    "%"
)


model_a_baseline["assignments"].to_csv(
    "model_a_baseline_assignments.csv",
    index=False
)

model_a_optimized["assignments"].to_csv(
    "model_a_optimized_assignments.csv",
    index=False
)

model_b_baseline["assignments"].to_csv(
    "model_b_baseline_assignments.csv",
    index=False
)

model_b_optimized["assignments"].to_csv(
    "model_b_optimized_assignments.csv",
    index=False
)

print()
print("=" * 70)
print("ASSIGNMENT FILES SAVED")
print("=" * 70)

print("model_a_baseline_assignments.csv")
print("model_a_optimized_assignments.csv")
print("model_b_baseline_assignments.csv")
print("model_b_optimized_assignments.csv")

FOUR-SCENARIO PULP OPTIMIZATION

Model_A | Baseline
Status              : Optimal
Wave objective      : ₹ 6268596257.9
Annualized ×50      : ₹ 313429812894.86
Open plants         : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

Model_A | Optimized
Status              : Optimal
Wave objective      : ₹ 5841004341.18
Annualized ×50      : ₹ 292050217058.8
Open plants         : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

Model_B | Baseline
Status              : Optimal
Wave objective      : ₹ 6268617377.52
Annualized ×50      : ₹ 313430868875.89
Open plants         : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

Model_B | Optimized
Status              : Optimal
Wave objective      : ₹ 5841132757.18
Annualized ×50      : ₹ 292056637858.97
Open plants         : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

MODEL A BASELINE VALIDATION
Calculated annualized : ₹ 313429812894.86
Previou

In [32]:
model_a_baseline = results[0]
model_b_baseline = results[2]

print("=" * 70)
print("BASELINE OBJECTIVE PRECISION CHECK")
print("=" * 70)

print(
    f"Model A baseline : ₹{model_a_baseline['annual_objective']:,.10f}"
)

print(
    f"Model B baseline : ₹{model_b_baseline['annual_objective']:,.10f}"
)

baseline_diff = (
    model_b_baseline["annual_objective"]
    - model_a_baseline["annual_objective"]
)

print(
    f"Difference       : ₹{baseline_diff:,.10f}"
)


print()
print("=" * 70)
print("MODEL A BASELINE COMPONENT RECONCILIATION")
print("=" * 70)

assignments = model_a_baseline["assignments"].copy()

new_baseline_fixed = sum(
    fixed_cost_wave[plant]
    for plant in model_a_baseline["open_plants"]
)

new_baseline_variable_wave = (
    assignments["variable_cost_wave"].sum()
)

new_baseline_freight_wave = (
    assignments["freight_cost_wave"].sum()
)

new_baseline_variable = (
    new_baseline_variable_wave
    * ANNUALIZATION_FACTOR
)

new_baseline_freight = (
    new_baseline_freight_wave
    * ANNUALIZATION_FACTOR
)

new_baseline_fixed = (
    new_baseline_fixed
    * ANNUALIZATION_FACTOR
)

new_baseline_total = (
    new_baseline_fixed
    + new_baseline_variable
    + new_baseline_freight
)

print(
    f"Fixed:    ₹{new_baseline_fixed:,.2f}"
)

print(
    f"Variable: ₹{new_baseline_variable:,.2f}"
)

print(
    f"Freight:  ₹{new_baseline_freight:,.2f}"
)

print(
    f"Total:    ₹{new_baseline_total:,.2f}"
)


print()
print("=" * 70)
print("OLD VS NEW COMPONENT COMPARISON")
print("=" * 70)

old_fixed = 37_603_800
old_variable = 313_249_972_805
old_freight = 667_143_117
old_total = 313_954_719_722

print(
    f"{'Component':<12}"
    f"{'Old':>20}"
    f"{'New':>20}"
    f"{'Difference':>20}"
)

print("-" * 72)

print(
    f"{'Fixed':<12}"
    f"₹{old_fixed:>19,.2f}"
    f"₹{new_baseline_fixed:>19,.2f}"
    f"₹{new_baseline_fixed - old_fixed:>19,.2f}"
)

print(
    f"{'Variable':<12}"
    f"₹{old_variable:>19,.2f}"
    f"₹{new_baseline_variable:>19,.2f}"
    f"₹{new_baseline_variable - old_variable:>19,.2f}"
)

print(
    f"{'Freight':<12}"
    f"₹{old_freight:>19,.2f}"
    f"₹{new_baseline_freight:>19,.2f}"
    f"₹{new_baseline_freight - old_freight:>19,.2f}"
)

print(
    f"{'Total':<12}"
    f"₹{old_total:>19,.2f}"
    f"₹{new_baseline_total:>19,.2f}"
    f"₹{new_baseline_total - old_total:>19,.2f}"
)


print()
print("=" * 70)
print("BASELINE ASSIGNMENT CHECK")
print("=" * 70)

assignment_map = (
    assignments
    .set_index("city")["Plant Code"]
    .to_dict()
)

print(
    "Chandigarh served by:",
    assignment_map.get("Chandigarh")
)

print(
    "Visakhapatnam served by:",
    assignment_map.get("Visakhapatnam")
)


print()
print("=" * 70)
print("CHANDIGARH / VISAKHAPATNAM DETAILS")
print("=" * 70)

print(
    assignments[
        assignments["city"].isin(
            ["Chandigarh", "Visakhapatnam"]
        )
    ][
        [
            "Plant Code",
            "city",
            "demand_orders_wave",
            "demand_units_day",
            "variable_cost_per_unit",
            "variable_cost_wave",
            "freight_cost_wave"
        ]
    ].to_string(index=False)
)

BASELINE OBJECTIVE PRECISION CHECK
Model A baseline : ₹313,429,812,894.8556518555
Model B baseline : ₹313,430,868,875.8903808594
Difference       : ₹1,055,981.0347290039

MODEL A BASELINE COMPONENT RECONCILIATION
Fixed:    ₹42,142,800.00
Variable: ₹313,249,972,804.67
Freight:  ₹137,697,290.18
Total:    ₹313,429,812,894.86

OLD VS NEW COMPONENT COMPARISON
Component                    Old                 New          Difference
------------------------------------------------------------------------
Fixed       ₹      37,603,800.00₹      42,142,800.00₹       4,539,000.00
Variable    ₹ 313,249,972,805.00₹ 313,249,972,804.67₹              -0.33
Freight     ₹     667,143,117.00₹     137,697,290.18₹    -529,445,826.82
Total       ₹ 313,954,719,722.00₹ 313,429,812,894.86₹    -524,906,827.14

BASELINE ASSIGNMENT CHECK
Chandigarh served by: PLANT03
Visakhapatnam served by: PLANT04

CHANDIGARH / VISAKHAPATNAM DETAILS
Plant Code          city  demand_orders_wave  demand_units_day  variable_cost_p

In [38]:
baseline_assignments = results[0]["assignments"].copy()

baseline_lanes = model_a_lanes.merge(
    baseline_assignments[["Plant Code", "city"]],
    on=["Plant Code", "city"],
    how="inner"
)

current_port = baseline_lanes["plant_port_cost_inr"].sum()
current_zone = baseline_lanes["plant_zone_cost_inr"].sum()
current_freight = baseline_lanes["total_lane_cost_inr"].sum()

old_freight_wave = 667143117 / 50

print("=" * 80)
print("HISTORICAL VS CURRENT BASELINE FREIGHT RECONCILIATION")
print("=" * 80)

print()
print(f"Current plant→port wave : ₹{current_port:,.2f}")
print(f"Current plant→zone wave : ₹{current_zone:,.2f}")
print(f"Current freight wave    : ₹{current_freight:,.2f}")

print()
print(f"Old freight wave        : ₹{old_freight_wave:,.2f}")

print()
print(f"Port difference         : ₹{current_port - old_freight_wave:,.2f}")
print(f"Zone difference         : ₹{current_zone:,.2f}")
print(f"Total current vs old    : ₹{current_freight - old_freight_wave:,.2f}")

print()
print("=" * 80)
print("IMPLIED OLD ZONE COST")
print("=" * 80)

implied_old_zone = old_freight_wave - current_port

print(
    f"Implied old plant→zone wave : ₹{implied_old_zone:,.2f}"
)

print(
    f"Current plant→zone wave     : ₹{current_zone:,.2f}"
)

print(
    f"Difference                   : ₹{implied_old_zone - current_zone:,.2f}"
)

print()
print("=" * 80)
print("IMPLIED OLD LAST-MILE RATE")
print("=" * 80)

rate_numerator = implied_old_zone

rate_denominator = (
    baseline_lanes["zone_weight_kg"] / 1000
    * baseline_lanes["distance_km"]
).sum()

implied_rate = rate_numerator / rate_denominator

print(
    f"Implied old rate : ₹{implied_rate:,.6f} per tonne-km"
)

print(
    f"Current rate     : ₹5.000000 per tonne-km"
)

HISTORICAL VS CURRENT BASELINE FREIGHT RECONCILIATION

Current plant→port wave : ₹1,001,292.70
Current plant→zone wave : ₹1,752,653.11
Current freight wave    : ₹2,753,945.80

Old freight wave        : ₹13,342,862.34

Port difference         : ₹-12,341,569.64
Zone difference         : ₹1,752,653.11
Total current vs old    : ₹-10,588,916.54

IMPLIED OLD ZONE COST
Implied old plant→zone wave : ₹12,341,569.64
Current plant→zone wave     : ₹1,752,653.11
Difference                   : ₹10,588,916.54

IMPLIED OLD LAST-MILE RATE
Implied old rate : ₹35.208250 per tonne-km
Current rate     : ₹5.000000 per tonne-km


In [39]:
print("=" * 80)
print("SEARCHING CURRENT NOTEBOOK VARIABLES FOR LAST-MILE RATE")
print("=" * 80)

for name in dir():
    if any(
        key in name.lower()
        for key in [
            "rate",
            "last_mile",
            "zone_cost",
            "transport",
            "tonne",
            "km"
        ]
    ):
        try:
            value = globals()[name]
            if isinstance(value, (int, float, np.integer, np.floating)):
                print(f"{name} = {value}")
        except:
            pass

SEARCHING CURRENT NOTEBOOK VARIABLES FOR LAST-MILE RATE
PORT_ZONE_RATE = 5
distance_km = 2066.9
implied_rate = 35.20824968410948
plant_zone_cost_inr = 515346.9405465883
rate_denominator = 350530.62150490214
rate_numerator = 12341569.643870672


In [42]:
print("=" * 80)
print("FIXED COST SOURCE CHECK")
print("=" * 80)

print(
    final_candidate_pool[
        [
            "Plant Code",
            "fixed_cost_annual",
            "footprint_sqft",
            "rent_per_sqft_month"
        ]
    ]
    .sort_values("Plant Code")
    .to_string(index=False)
)

FIXED COST SOURCE CHECK
Plant Code  fixed_cost_annual  footprint_sqft  rent_per_sqft_month
   PLANT01           22470000           53500                   35
   PLANT02            1490400            6900                   18
   PLANT03           21273000           50650                   35
   PLANT04            5983200           27700                   18
   PLANT05            8085000           19250                   35
   PLANT06             529200            2450                   18
   PLANT07            2862000           13250                   18
   PLANT08             151200             700                   18
   PLANT09             118800             550                   18
   PLANT10            2478000            5900                   35
   PLANT11            3585600           16600                   18
   PLANT12            4389000           10450                   35
   PLANT13            5292000           24500                   18
   PLANT14            5929200         

In [45]:
print("=" * 90)
print("MODEL A OPTIMIZED — PLANT CAPACITY UTILIZATION")
print("=" * 90)

optimized = results[1]["assignments"].copy()

capacity_check = (
    optimized
    .groupby("Plant Code", as_index=False)
    .agg(
        assigned_orders=("demand_orders_wave", "sum"),
        assigned_units_wave=("demand_units_wave", "sum")
    )
)

capacity_check["capacity_wave"] = (
    capacity_check["Plant Code"]
    .map(capacity_wave)
)

capacity_check["capacity_utilization_pct"] = (
    capacity_check["assigned_orders"]
    / capacity_check["capacity_wave"]
    * 100
)

capacity_check["unused_capacity_wave"] = (
    capacity_check["capacity_wave"]
    - capacity_check["assigned_orders"]
)

capacity_check = capacity_check.sort_values(
    "capacity_utilization_pct",
    ascending=False
)

print(
    capacity_check.to_string(index=False)
)

print()
print("=" * 90)
print("CAPACITY VALIDATION")
print("=" * 90)

print(
    "Total assigned orders :",
    capacity_check["assigned_orders"].sum()
)

print(
    "Total plant capacity  :",
    capacity_check["capacity_wave"].sum()
)

print(
    "Maximum utilization   :",
    round(
        capacity_check["capacity_utilization_pct"].max(),
        2
    ),
    "%"
)

print(
    "All capacity feasible :",
    (
        capacity_check["assigned_orders"]
        <= capacity_check["capacity_wave"] + 1e-9
    ).all()
)

MODEL A OPTIMIZED — PLANT CAPACITY UTILIZATION
Plant Code  assigned_orders  assigned_units_wave  capacity_wave  capacity_utilization_pct  unused_capacity_wave
   PLANT04           2654.0           57457885.0         2770.0                 95.812274                 116.0
   PLANT07           1256.0           60205370.0         1325.0                 94.792453                  69.0
   PLANT13           2281.0           14042725.0         2450.0                 93.102041                 169.0
   PLANT02            405.0            2959655.0          690.0                 58.695652                 285.0
   PLANT05           1115.0            7447135.0         1925.0                 57.922078                 810.0
   PLANT03           1504.0            5453805.0         5065.0                 29.693978                3561.0

CAPACITY VALIDATION
Total assigned orders : 9215.0
Total plant capacity  : 14225.0
Maximum utilization   : 95.81 %
All capacity feasible : True


In [46]:
print("=" * 100)
print("MODEL A OPTIMIZED — ZONE ASSIGNMENTS")
print("=" * 100)

optimized_assignments = results[1]["assignments"].copy()

optimized_assignments["capacity_wave"] = (
    optimized_assignments["Plant Code"]
    .map(capacity_wave)
)

optimized_assignments["variable_cost_per_unit"] = (
    optimized_assignments["Plant Code"]
    .map(variable_cost_per_unit)
)

optimized_assignments["freight_cost_wave"] = 0.0

for idx, row in optimized_assignments.iterrows():
    plant = row["Plant Code"]
    zone = row["city"]

    match = model_a_lanes[
        (model_a_lanes["Plant Code"] == plant) &
        (model_a_lanes["city"] == zone)
    ]

    if not match.empty:
        optimized_assignments.loc[
            idx,
            "freight_cost_wave"
        ] = match.iloc[0]["total_lane_cost_inr"]

optimized_assignments["annual_freight_cost"] = (
    optimized_assignments["freight_cost_wave"] * 50
)

optimized_assignments["capacity_utilization_pct"] = (
    optimized_assignments["Plant Code"]
    .map(
        capacity_check.set_index("Plant Code")[
            "capacity_utilization_pct"
        ]
    )
)

print(
    optimized_assignments[
        [
            "Plant Code",
            "city",
            "demand_orders_wave",
            "demand_units_wave",
            "variable_cost_per_unit",
            "freight_cost_wave",
            "annual_freight_cost",
            "capacity_utilization_pct"
        ]
    ]
    .sort_values(["Plant Code", "city"])
    .to_string(index=False)
)

optimized_assignments.to_csv(
    "model_a_optimized_detailed_assignments.csv",
    index=False
)

print()
print("=" * 100)
print("FILE SAVED")
print("=" * 100)
print("model_a_optimized_detailed_assignments.csv")

MODEL A OPTIMIZED — ZONE ASSIGNMENTS
Plant Code          city  demand_orders_wave  demand_units_wave  variable_cost_per_unit  freight_cost_wave  annual_freight_cost  capacity_utilization_pct
   PLANT02        Indore               185.0          1531355.0               45.362834       41412.958615         2.070648e+06                 58.695652
   PLANT02        Jaipur               220.0          1428300.0               45.362834       17067.332861         8.533666e+05                 58.695652
   PLANT03     Delhi NCR              1504.0          5453805.0               49.162680      108586.841190         5.429342e+06                 29.693978
   PLANT04     Bengaluru               368.0          4119250.0               40.707813      131003.013609         6.550151e+06                 95.812274
   PLANT04        Bhopal               592.0         27608425.0               40.707813      115594.022009         5.779701e+06                 95.812274
   PLANT04    Coimbatore               

In [47]:
print("=" * 100)
print("MODEL A — SINGLE-PLANT FAILURE ANALYSIS")
print("=" * 100)

optimized_plants = results[1]["open_plants"]

disruption_results = []

for failed_plant in optimized_plants:

    available_plants = [
        p for p in plants
        if p != failed_plant
    ]

    lane_lookup = {}

    for _, row in model_a_lanes.iterrows():

        plant = row["Plant Code"]
        zone = row["city"]

        if plant not in available_plants:
            continue

        if not bool(row["is_feasible"]):
            continue

        lane_lookup[(plant, zone)] = float(
            row["total_lane_cost_inr"]
        )

    problem = pulp.LpProblem(
        f"Failure_{failed_plant}",
        pulp.LpMinimize
    )

    y = {
        i: pulp.LpVariable(
            f"fail_y_{i}",
            cat="Binary"
        )
        for i in available_plants
    }

    z = {
        (i, j): pulp.LpVariable(
            f"fail_z_{i}_{j}",
            cat="Binary"
        )
        for (i, j) in lane_lookup
    }

    objective_terms = []

    for i in available_plants:
        objective_terms.append(
            fixed_cost_wave[i] * y[i]
        )

    for (i, j), var in z.items():

        variable_cost_wave = (
            variable_cost_per_unit[i]
            * zone_units_day[j]
            * PLANNING_DAYS
        )

        freight_cost_wave = lane_lookup[(i, j)]

        objective_terms.append(
            (
                variable_cost_wave
                + freight_cost_wave
            ) * var
        )

    problem += pulp.lpSum(objective_terms)

    for j in zones:

        feasible_plants = [
            i
            for i in available_plants
            if (i, j) in z
        ]

        problem += (
            pulp.lpSum(
                z[(i, j)]
                for i in feasible_plants
            ) == 1
        )

    for i in available_plants:

        assigned_orders = [
            zone_orders[j] * z[(i, j)]
            for j in zones
            if (i, j) in z
        ]

        problem += (
            pulp.lpSum(assigned_orders)
            <= capacity_wave[i] * y[i]
        )

    for (i, j), var in z.items():

        problem += (
            var <= y[i]
        )

    problem.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    status = pulp.LpStatus[problem.status]

    if status == "Optimal":

        annual_cost = (
            pulp.value(problem.objective)
            * ANNUALIZATION_FACTOR
        )

        open_plants_failure = [
            i
            for i in available_plants
            if pulp.value(y[i]) > 0.5
        ]

        disruption_results.append({
            "failed_plant": failed_plant,
            "status": status,
            "annual_cost": annual_cost,
            "cost_increase": annual_cost - results[1]["annual_objective"],
            "open_plants": open_plants_failure
        })

    else:

        disruption_results.append({
            "failed_plant": failed_plant,
            "status": status,
            "annual_cost": np.nan,
            "cost_increase": np.nan,
            "open_plants": []
        })

disruption_df = pd.DataFrame(
    disruption_results
)

print()

print(
    disruption_df[
        [
            "failed_plant",
            "status",
            "annual_cost",
            "cost_increase",
            "open_plants"
        ]
    ].to_string(index=False)
)

print()
print("=" * 100)
print("RESILIENCE SUMMARY")
print("=" * 100)

for _, row in disruption_df.iterrows():

    if row["status"] == "Optimal":

        pct_increase = (
            row["cost_increase"]
            / results[1]["annual_objective"]
            * 100
        )

        print(
            f'{row["failed_plant"]}: '
            f'Feasible | '
            f'Cost increase ₹{row["cost_increase"]:,.2f} '
            f'({pct_increase:.2f}%)'
        )

    else:

        print(
            f'{row["failed_plant"]}: '
            f'INFEASIBLE'
        )

MODEL A — SINGLE-PLANT FAILURE ANALYSIS

failed_plant  status  annual_cost  cost_increase                                            open_plants
     PLANT05 Optimal 2.930766e+11   1.026344e+09          [PLANT02, PLANT07, PLANT03, PLANT13, PLANT04]
     PLANT02 Optimal 2.921978e+11   1.475340e+08          [PLANT05, PLANT07, PLANT03, PLANT13, PLANT04]
     PLANT07 Optimal 3.111332e+11   1.908298e+10 [PLANT05, PLANT02, PLANT10, PLANT03, PLANT13, PLANT04]
     PLANT03 Optimal 2.925612e+11   5.109560e+08 [PLANT05, PLANT02, PLANT10, PLANT07, PLANT13, PLANT04]
     PLANT13 Optimal 2.943623e+11   2.312118e+09 [PLANT05, PLANT02, PLANT10, PLANT07, PLANT03, PLANT04]
     PLANT04 Optimal 3.058007e+11   1.375043e+10 [PLANT05, PLANT02, PLANT10, PLANT07, PLANT03, PLANT13]

RESILIENCE SUMMARY
PLANT05: Feasible | Cost increase ₹1,026,344,115.91 (0.35%)
PLANT02: Feasible | Cost increase ₹147,534,010.47 (0.05%)
PLANT07: Feasible | Cost increase ₹19,082,978,291.67 (6.53%)
PLANT03: Feasible | Cost increas

In [58]:
failure_results = []

optimized_plants = results[1]["open_plants"]
optimized_cost = results[1]["annual_objective"]

for failed_plant in optimized_plants:

    remaining_plants = [
        p for p in plants
        if p != failed_plant
    ]

    lane_lookup = {}

    for _, row in model_a_lanes.iterrows():

        plant = row["Plant Code"]
        zone = row["city"]

        if not bool(row["is_feasible"]):
            continue

        if plant == failed_plant:
            continue

        lane_lookup[(plant, zone)] = float(
            row["total_lane_cost_inr"]
        )

    problem = pulp.LpProblem(
        f"Failure_{failed_plant}",
        pulp.LpMinimize
    )

    y = {
        i: pulp.LpVariable(
            f"failure_y_{i}",
            cat="Binary"
        )
        for i in remaining_plants
    }

    z = {
        (i, j): pulp.LpVariable(
            f"failure_z_{i}_{j}",
            cat="Binary"
        )
        for (i, j) in lane_lookup
    }

    objective_terms = []

    for i in remaining_plants:

        objective_terms.append(
            fixed_cost_wave[i] * y[i]
        )

    for (i, j), var in z.items():

        variable_wave = (
            variable_cost_per_unit[i]
            * zone_units_day[j]
            * PLANNING_DAYS
        )

        freight_wave = lane_lookup[(i, j)]

        objective_terms.append(
            (
                variable_wave
                + freight_wave
            ) * var
        )

    problem += pulp.lpSum(objective_terms)

    for j in zones:

        feasible_plants = [
            i
            for i in remaining_plants
            if (i, j) in z
        ]

        problem += (
            pulp.lpSum(
                z[(i, j)]
                for i in feasible_plants
            ) == 1,
            f"failure_single_source_{j}"
        )

    for i in remaining_plants:

        assigned_orders = [
            zone_orders[j] * z[(i, j)]
            for j in zones
            if (i, j) in z
        ]

        problem += (
            pulp.lpSum(assigned_orders)
            <= capacity_wave[i] * y[i],
            f"failure_capacity_{i}"
        )

    for (i, j), var in z.items():

        problem += (
            var <= y[i],
            f"failure_open_link_{i}_{j}"
        )

    problem.solve(
        pulp.PULP_CBC_CMD(msg=False)
    )

    status = pulp.LpStatus[problem.status]

    if status != "Optimal":

        failure_results.append({
            "failed_plant": failed_plant,
            "status": status,
            "annual_cost": np.nan,
            "cost_increase": np.nan,
            "cost_increase_pct": np.nan,
            "open_plants": [],
            "reassigned_zones": 0,
            "assignments": pd.DataFrame()
        })

        continue

    wave_objective = pulp.value(
        problem.objective
    )

    annual_cost = (
        wave_objective
        * ANNUALIZATION_FACTOR
    )

    open_plants_failure = [
        i
        for i in remaining_plants
        if pulp.value(y[i]) > 0.5
    ]

    assignment_records = []

    for (i, j), var in z.items():

        if pulp.value(var) > 0.5:

            assignment_records.append({
                "Plant Code": i,
                "city": j,
                "demand_orders_wave": zone_orders[j],
                "demand_units_wave":
                    zone_units_day[j]
                    * PLANNING_DAYS
            })

    failure_assignments = pd.DataFrame(
        assignment_records
    )

    assigned_zones = (
        failure_assignments["city"]
        .nunique()
    )

    original_assignments = (
        results[1]["assignments"]
        .set_index("city")["Plant Code"]
        .to_dict()
    )

    failure_assignment_map = (
        failure_assignments
        .set_index("city")["Plant Code"]
        .to_dict()
    )

    reassigned_zones = sum(
        original_assignments.get(zone)
        != failure_assignment_map.get(zone)
        for zone in zones
    )

    cost_increase = (
        annual_cost
        - optimized_cost
    )

    cost_increase_pct = (
        cost_increase
        / optimized_cost
        * 100
    )

    failure_results.append({
        "failed_plant": failed_plant,
        "status": status,
        "annual_cost": annual_cost,
        "cost_increase": cost_increase,
        "cost_increase_pct": cost_increase_pct,
        "open_plants": open_plants_failure,
        "reassigned_zones": reassigned_zones,
        "assignments": failure_assignments
    })


print("=" * 100)
print("MODEL A — SINGLE-PLANT FAILURE ANALYSIS")
print("=" * 100)

failure_summary = []

for r in failure_results:

    failure_summary.append({
        "failed_plant": r["failed_plant"],
        "status": r["status"],
        "annual_cost": r["annual_cost"],
        "cost_increase": r["cost_increase"],
        "cost_increase_pct": r["cost_increase_pct"],
        "reassigned_zones": r["reassigned_zones"],
        "open_plants": r["open_plants"]
    })

failure_summary_df = pd.DataFrame(
    failure_summary
)

print(
    failure_summary_df[
        [
            "failed_plant",
            "status",
            "annual_cost",
            "cost_increase",
            "cost_increase_pct",
            "reassigned_zones",
            "open_plants"
        ]
    ].to_string(index=False)
)

print()
print("=" * 100)
print("FAILURE VALIDATION")
print("=" * 100)

for r in failure_results:

    if r["status"] == "Optimal":

        zones_served = (
            r["assignments"]["city"]
            .nunique()
        )

        orders_served = (
            r["assignments"]["demand_orders_wave"]
            .sum()
        )

        failed_still_open = (
            r["failed_plant"]
            in r["open_plants"]
        )

        print(
            f'{r["failed_plant"]}: '
            f'Status={r["status"]} | '
            f'Zones={zones_served}/{len(zones)} | '
            f'Orders={orders_served:.0f} | '
            f'Failed plant open={failed_still_open}'
        )

print()
print("=" * 100)
print("RESILIENCE SUMMARY")
print("=" * 100)

for r in failure_results:

    if r["status"] == "Optimal":

        print(
            f'{r["failed_plant"]}: '
            f'Feasible | '
            f'Cost increase ₹{r["cost_increase"]:,.2f} '
            f'({r["cost_increase_pct"]:.2f}%) | '
            f'Reassigned zones: {r["reassigned_zones"]}'
        )

    else:

        print(
            f'{r["failed_plant"]}: '
            f'INFEASIBLE'
        )

failure_summary_df.to_csv(
    processed_path / "model_a_resilience_analysis.csv",
    index=False
)

print()
print("=" * 100)
print("FILE SAVED")
print("=" * 100)
print("model_a_resilience_analysis.csv")

MODEL A — SINGLE-PLANT FAILURE ANALYSIS
failed_plant  status  annual_cost  cost_increase  cost_increase_pct  reassigned_zones                                            open_plants
     PLANT05 Optimal 2.930766e+11   1.026344e+09           0.351427                 1          [PLANT02, PLANT07, PLANT03, PLANT13, PLANT04]
     PLANT02 Optimal 2.921978e+11   1.475340e+08           0.050517                 2          [PLANT05, PLANT07, PLANT03, PLANT13, PLANT04]
     PLANT07 Optimal 3.111332e+11   1.908298e+10           6.534143                13 [PLANT05, PLANT02, PLANT10, PLANT03, PLANT13, PLANT04]
     PLANT03 Optimal 2.925612e+11   5.109560e+08           0.174955                12 [PLANT05, PLANT02, PLANT10, PLANT07, PLANT13, PLANT04]
     PLANT13 Optimal 2.943623e+11   2.312118e+09           0.791685                 5 [PLANT05, PLANT02, PLANT10, PLANT07, PLANT03, PLANT04]
     PLANT04 Optimal 3.058007e+11   1.375043e+10           4.708243                10 [PLANT05, PLANT02, PLANT10, 

In [50]:
optimized = results[1]

baseline_assignments = (
    optimized["assignments"]
    .set_index("city")["Plant Code"]
    .to_dict()
)

print("=" * 100)
print("MODEL A — CRITICAL PLANT FAILURE ZONE REASSIGNMENT")
print("=" * 100)

for failed_plant in ["PLANT07", "PLANT04"]:

    failure_result = next(
        r for r in failure_results
        if r["failed_plant"] == failed_plant
    )

    failure_assignments = (
        failure_result["assignments"]
        .set_index("city")["Plant Code"]
        .to_dict()
    )

    changed_zones = []

    for zone in zones:

        original = baseline_assignments.get(zone)
        new = failure_assignments.get(zone)

        if original != new:

            changed_zones.append({
                "city": zone,
                "original_plant": original,
                "failure_plant": new
            })

    print()
    print("=" * 100)
    print(f"FAILED PLANT: {failed_plant}")
    print("=" * 100)

    print(
        "Number of reassigned zones:",
        len(changed_zones)
    )

    if changed_zones:

        print(
            pd.DataFrame(changed_zones)
            .sort_values("city")
            .to_string(index=False)
        )

    else:

        print("No zone reassignment.")

MODEL A — CRITICAL PLANT FAILURE ZONE REASSIGNMENT

FAILED PLANT: PLANT07
Number of reassigned zones: 13
         city original_plant failure_plant
    Ahmedabad        PLANT07       PLANT04
    Bengaluru        PLANT04       PLANT13
   Chandigarh        PLANT13       PLANT05
      Chennai        PLANT05       PLANT13
     Guwahati        PLANT07       PLANT04
    Hyderabad        PLANT07       PLANT04
       Indore        PLANT02       PLANT13
       Jaipur        PLANT02       PLANT10
        Kochi        PLANT04       PLANT13
      Kolkata        PLANT13       PLANT02
       Mumbai        PLANT07       PLANT04
         Pune        PLANT07       PLANT04
Visakhapatnam        PLANT07       PLANT04

FAILED PLANT: PLANT04
Number of reassigned zones: 10
      city original_plant failure_plant
 Bengaluru        PLANT04       PLANT13
    Bhopal        PLANT04       PLANT13
Chandigarh        PLANT13       PLANT05
   Chennai        PLANT05       PLANT03
Coimbatore        PLANT04       PLANT13

In [51]:
resilience_df = pd.DataFrame([
    {
        "failed_plant": r["failed_plant"],
        "status": r["status"],
        "annual_cost": r["annual_cost"],
        "cost_increase": r["cost_increase"],
        "cost_increase_pct": (
            r["cost_increase"]
            / results[1]["annual_objective"]
            * 100
            if r["status"] == "Optimal"
            else np.nan
        ),
        "reassigned_zones": (
            sum(
                baseline_assignments.get(zone)
                != r["assignments"]
                .set_index("city")["Plant Code"]
                .to_dict()
                .get(zone)
                for zone in zones
            )
            if r["status"] == "Optimal"
            else np.nan
        )
    }
    for r in failure_results
])

resilience_df = resilience_df.sort_values(
    "cost_increase",
    ascending=False
).reset_index(drop=True)

print("=" * 100)
print("FINAL RESILIENCE RANKING")
print("=" * 100)

print(
    resilience_df.to_string(index=False)
)

resilience_df.to_csv(
    processed_path / "model_a_resilience_analysis.csv",
    index=False
)

print()
print("Saved:")
print("model_a_resilience_analysis.csv")

FINAL RESILIENCE RANKING
failed_plant  status  annual_cost  cost_increase  cost_increase_pct  reassigned_zones
     PLANT07 Optimal 3.111332e+11   1.908298e+10           6.534143                13
     PLANT04 Optimal 3.058007e+11   1.375043e+10           4.708243                10
     PLANT13 Optimal 2.943623e+11   2.312118e+09           0.791685                 5
     PLANT05 Optimal 2.930766e+11   1.026344e+09           0.351427                 1
     PLANT03 Optimal 2.925612e+11   5.109560e+08           0.174955                12
     PLANT02 Optimal 2.921978e+11   1.475340e+08           0.050517                 2

Saved:
model_a_resilience_analysis.csv


In [52]:
print("=" * 90)
print("FINAL PROJECT KPI EXTRACTION")
print("=" * 90)

model_a_baseline = results[0]
model_a_optimized = results[1]
model_b_baseline = results[2]
model_b_optimized = results[3]

kpi_summary = pd.DataFrame([
    {
        "Model": "Model A",
        "Scenario": "Baseline",
        "Annual_Cost_INR": model_a_baseline["annual_objective"],
        "Open_Plants": len(model_a_baseline["open_plants"]),
        "Zones": len(model_a_baseline["assignments"])
    },
    {
        "Model": "Model A",
        "Scenario": "Optimized",
        "Annual_Cost_INR": model_a_optimized["annual_objective"],
        "Open_Plants": len(model_a_optimized["open_plants"]),
        "Zones": len(model_a_optimized["assignments"])
    },
    {
        "Model": "Model B",
        "Scenario": "Baseline",
        "Annual_Cost_INR": model_b_baseline["annual_objective"],
        "Open_Plants": len(model_b_baseline["open_plants"]),
        "Zones": len(model_b_baseline["assignments"])
    },
    {
        "Model": "Model B",
        "Scenario": "Optimized",
        "Annual_Cost_INR": model_b_optimized["annual_objective"],
        "Open_Plants": len(model_b_optimized["open_plants"]),
        "Zones": len(model_b_optimized["assignments"])
    }
])

kpi_summary["Savings_INR"] = np.nan
kpi_summary["Savings_Pct"] = np.nan

kpi_summary.loc[
    (kpi_summary["Model"] == "Model A") &
    (kpi_summary["Scenario"] == "Optimized"),
    "Savings_INR"
] = (
    model_a_baseline["annual_objective"]
    - model_a_optimized["annual_objective"]
)

kpi_summary.loc[
    (kpi_summary["Model"] == "Model A") &
    (kpi_summary["Scenario"] == "Optimized"),
    "Savings_Pct"
] = (
    (
        model_a_baseline["annual_objective"]
        - model_a_optimized["annual_objective"]
    )
    / model_a_baseline["annual_objective"]
    * 100
)

kpi_summary.loc[
    (kpi_summary["Model"] == "Model B") &
    (kpi_summary["Scenario"] == "Optimized"),
    "Savings_INR"
] = (
    model_b_baseline["annual_objective"]
    - model_b_optimized["annual_objective"]
)

kpi_summary.loc[
    (kpi_summary["Model"] == "Model B") &
    (kpi_summary["Scenario"] == "Optimized"),
    "Savings_Pct"
] = (
    (
        model_b_baseline["annual_objective"]
        - model_b_optimized["annual_objective"]
    )
    / model_b_baseline["annual_objective"]
    * 100
)

print()
print(kpi_summary.to_string(index=False))

kpi_summary.to_csv(
    processed_path / "network_optimization_kpis.csv",
    index=False
)

print()
print("Saved:")
print("network_optimization_kpis.csv")

FINAL PROJECT KPI EXTRACTION

  Model  Scenario  Annual_Cost_INR  Open_Plants  Zones  Savings_INR  Savings_Pct
Model A  Baseline     3.134298e+11            7     18          NaN          NaN
Model A Optimized     2.920502e+11            6     18 2.137960e+10     6.821175
Model B  Baseline     3.134309e+11            7     18          NaN          NaN
Model B Optimized     2.920566e+11            6     18 2.137423e+10     6.819440

Saved:
network_optimization_kpis.csv


In [53]:
master_results = []

for result in results:

    master_results.append({
        "Model": result["model"],
        "Scenario": result["scenario"],
        "Annual_Cost_INR": result["annual_objective"],
        "Open_Plant_Count": len(result["open_plants"]),
        "Open_Plants": ", ".join(result["open_plants"]),
        "Zone_Count": len(result["assignments"])
    })

master_results_df = pd.DataFrame(master_results)

resilience_master = resilience_df.copy()

optimized_capacity = pd.DataFrame()

for plant in model_a_optimized["open_plants"]:

    assigned_orders = (
        model_a_optimized["assignments"]
        .loc[
            model_a_optimized["assignments"]["Plant Code"] == plant,
            "demand_orders_wave"
        ]
        .sum()
    )

    capacity = capacity_wave[plant]

    optimized_capacity = pd.concat([
        optimized_capacity,
        pd.DataFrame([{
            "Plant Code": plant,
            "Assigned_Orders": assigned_orders,
            "Capacity_Wave": capacity,
            "Capacity_Utilization_Pct":
                assigned_orders / capacity * 100,
            "Unused_Capacity":
                capacity - assigned_orders
        }])
    ], ignore_index=True)

model_comparison = pd.DataFrame([
    {
        "Metric": "Baseline Annual Cost",
        "Model_A": model_a_baseline["annual_objective"],
        "Model_B": model_b_baseline["annual_objective"]
    },
    {
        "Metric": "Optimized Annual Cost",
        "Model_A": model_a_optimized["annual_objective"],
        "Model_B": model_b_optimized["annual_objective"]
    },
    {
        "Metric": "Annual Savings",
        "Model_A":
            model_a_baseline["annual_objective"]
            - model_a_optimized["annual_objective"],
        "Model_B":
            model_b_baseline["annual_objective"]
            - model_b_optimized["annual_objective"]
    },
    {
        "Metric": "Savings %",
        "Model_A":
            (
                (
                    model_a_baseline["annual_objective"]
                    - model_a_optimized["annual_objective"]
                )
                / model_a_baseline["annual_objective"]
                * 100
            ),
        "Model_B":
            (
                (
                    model_b_baseline["annual_objective"]
                    - model_b_optimized["annual_objective"]
                )
                / model_b_baseline["annual_objective"]
                * 100
            )
    }
])

master_results_df.to_csv(
    processed_path / "final_network_results.csv",
    index=False
)

optimized_capacity.to_csv(
    processed_path / "optimized_capacity_utilization.csv",
    index=False
)

model_comparison.to_csv(
    processed_path / "model_a_vs_model_b.csv",
    index=False
)

print("=" * 90)
print("FINAL PROJECT OUTPUTS")
print("=" * 90)

print()
print("Network results:")
print("final_network_results.csv")

print()
print("Capacity utilization:")
print("optimized_capacity_utilization.csv")

print()
print("Model comparison:")
print("model_a_vs_model_b.csv")

print()
print("Resilience:")
print("model_a_resilience_analysis.csv")

print()
print("=" * 90)
print("OPTIMIZED MODEL A PLANTS")
print("=" * 90)

print(model_a_optimized["open_plants"])

print()
print("=" * 90)
print("MODEL A SAVINGS")
print("=" * 90)

savings_a = (
    model_a_baseline["annual_objective"]
    - model_a_optimized["annual_objective"]
)

print(f"Annual savings : ₹{savings_a:,.2f}")
print(
    f"Savings %     : "
    f"{savings_a / model_a_baseline['annual_objective'] * 100:.2f}%"
)

print()
print("=" * 90)
print("RESILIENCE PRIORITY")
print("=" * 90)

print(
    resilience_master[
        [
            "failed_plant",
            "cost_increase",
            "cost_increase_pct",
            "reassigned_zones"
        ]
    ].to_string(index=False)
)

FINAL PROJECT OUTPUTS

Network results:
final_network_results.csv

Capacity utilization:
optimized_capacity_utilization.csv

Model comparison:
model_a_vs_model_b.csv

Resilience:
model_a_resilience_analysis.csv

OPTIMIZED MODEL A PLANTS
['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

MODEL A SAVINGS
Annual savings : ₹21,379,595,836.05
Savings %     : 6.82%

RESILIENCE PRIORITY
failed_plant  cost_increase  cost_increase_pct  reassigned_zones
     PLANT07   1.908298e+10           6.534143                13
     PLANT04   1.375043e+10           4.708243                10
     PLANT13   2.312118e+09           0.791685                 5
     PLANT05   1.026344e+09           0.351427                 1
     PLANT03   5.109560e+08           0.174955                12
     PLANT02   1.475340e+08           0.050517                 2


In [54]:
print("=" * 100)
print("FINAL FREIGHT RATE RECONCILIATION")
print("=" * 100)

baseline_assignments = results[0]["assignments"].copy()

baseline_lane_details = model_a_lanes.merge(
    baseline_assignments[["Plant Code", "city"]],
    on=["Plant Code", "city"],
    how="inner"
)

baseline_lane_details["current_zone_cost"] = (
    baseline_lane_details["plant_zone_cost_inr"]
)

baseline_lane_details["implied_old_zone_cost"] = (
    baseline_lane_details["zone_weight_kg"] / 1000
    * baseline_lane_details["distance_km"]
    * 35.20824968410948
)

baseline_lane_details["current_rate"] = 5.0
baseline_lane_details["implied_old_rate"] = 35.20824968410948

print()
print(
    baseline_lane_details[
        [
            "Plant Code",
            "city",
            "zone_weight_kg",
            "distance_km",
            "plant_zone_cost_inr",
            "implied_old_zone_cost"
        ]
    ]
    .sort_values(["Plant Code", "city"])
    .to_string(index=False)
)

print()
print("=" * 100)
print("ZONE COST TOTALS")
print("=" * 100)

current_zone = baseline_lane_details[
    "plant_zone_cost_inr"
].sum()

implied_old_zone = baseline_lane_details[
    "implied_old_zone_cost"
].sum()

print(
    f"Current zone cost, wave : ₹{current_zone:,.2f}"
)

print(
    f"Implied old zone, wave  : ₹{implied_old_zone:,.2f}"
)

print(
    f"Difference              : ₹{implied_old_zone - current_zone:,.2f}"
)

print()
print("=" * 100)
print("RATE CHECK")
print("=" * 100)

print("Current rate : ₹5.000000 / tonne-km")
print(
    f"Implied old rate : ₹{35.20824968410948:.6f} / tonne-km"
)

print()
print(
    "Old/current rate ratio :",
    round(
        35.20824968410948 / 5,
        4
    )
)

FINAL FREIGHT RATE RECONCILIATION

Plant Code          city  zone_weight_kg  distance_km  plant_zone_cost_inr  implied_old_zone_cost
   PLANT03    Chandigarh    29329.459590       1903.5        279143.131652           1.965628e+06
   PLANT03     Delhi NCR     8622.451359       1713.1         73855.607116           5.200653e+05
   PLANT03        Jaipur     1554.118818       1765.1         13715.875630           9.658239e+04
   PLANT03       Kolkata     7298.371246          0.0             0.000000           0.000000e+00
   PLANT04     Ahmedabad     5343.798448       1239.3         33112.847085           2.331691e+05
   PLANT04        Bhopal     9430.857633       1083.4         51086.955798           3.597365e+05
   PLANT04    Coimbatore    16571.403311       2850.2        236159.068586           1.662949e+06
   PLANT04      Guwahati      446.572337       2011.5          4491.401275           3.162688e+04
   PLANT04     Hyderabad     1466.215182       1942.6         14241.348060         

In [55]:
print("=" * 100)
print("FAILURE ANALYSIS VALIDATION")
print("=" * 100)

print()

for r in failure_results:

    assignments = r["assignments"]

    assigned_zones = (
        assignments["city"].nunique()
        if not assignments.empty
        else 0
    )

    assigned_orders = (
        assignments["demand_orders_wave"].sum()
        if not assignments.empty
        else 0
    )

    print(
        f'{r["failed_plant"]}: '
        f'Status={r["status"]} | '
        f'Zones={assigned_zones}/{len(zones)} | '
        f'Orders={assigned_orders:.0f}'
    )

FAILURE ANALYSIS VALIDATION

PLANT05: Status=Optimal | Zones=18/18 | Orders=9215
PLANT02: Status=Optimal | Zones=18/18 | Orders=9215
PLANT07: Status=Optimal | Zones=18/18 | Orders=9215
PLANT03: Status=Optimal | Zones=18/18 | Orders=9215
PLANT13: Status=Optimal | Zones=18/18 | Orders=9215
PLANT04: Status=Optimal | Zones=18/18 | Orders=9215


In [56]:
print("=" * 90)
print("SEARCHING NOTEBOOK DATA FOR FREIGHT RATE ASSUMPTIONS")
print("=" * 90)

print()

print("PORT_ZONE_RATE =", PORT_ZONE_RATE)

print()

for name in [
    "freight_rates",
    "plant_zone_distance",
    "customer_demand_regional"
]:

    if name in globals():

        obj = globals()[name]

        print(f"{name}:")
        print("shape:", getattr(obj, "shape", "N/A"))

        if hasattr(obj, "columns"):
            print("columns:", obj.columns.tolist())

        print()

SEARCHING NOTEBOOK DATA FOR FREIGHT RATE ASSUMPTIONS

PORT_ZONE_RATE = 5

freight_rates:
shape: (1540, 11)
columns: ['Carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd', 'minimum cost', 'rate', 'mode_dsc', 'tpt_day_cnt', 'Carrier type']

plant_zone_distance:
shape: (288, 3)
columns: ['Plant Code', 'city', 'distance_km']

customer_demand_regional:
shape: (18, 7)
columns: ['city', 'total_units_day', 'total_weight_day', 'order_count', 'customer_count', 'latitude', 'longitude']



In [57]:
baseline_assignments = results[0]["assignments"].copy()

print("=" * 90)
print("MODEL A LANE-COST STRUCTURE CHECK")
print("=" * 90)

print()
print("Model A lane-cost columns:")
print(model_a_lanes.columns.tolist())

print()
print("Model A lane-cost summary:")
print(
    model_a_lanes[
        [
            "zone_weight_kg",
            "distance_km",
            "plant_port_cost_inr",
            "plant_zone_cost_inr",
            "total_lane_cost_inr"
        ]
    ].describe().to_string()
)

print()
print("=" * 90)
print("BASELINE ASSIGNMENT LANE BREAKDOWN")
print("=" * 90)

baseline_lane_details = model_a_lanes.merge(
    baseline_assignments[["Plant Code", "city"]],
    on=["Plant Code", "city"],
    how="inner"
)

print(
    baseline_lane_details[
        [
            "Plant Code",
            "city",
            "zone_weight_kg",
            "distance_km",
            "plant_port_cost_inr",
            "plant_zone_cost_inr",
            "total_lane_cost_inr"
        ]
    ]
    .sort_values(["Plant Code", "city"])
    .to_string(index=False)
)

print()
print("=" * 90)
print("LAST-MILE COST CHECK")
print("=" * 90)

print(
    "Rows with zero plant→zone cost:",
    (baseline_lane_details["plant_zone_cost_inr"] == 0).sum()
)

print(
    "Rows with positive plant→zone cost:",
    (baseline_lane_details["plant_zone_cost_inr"] > 0).sum()
)

print(
    "Total plant→port wave cost : ₹",
    round(
        baseline_lane_details["plant_port_cost_inr"].sum(),
        2
    )
)

print(
    "Total plant→zone wave cost : ₹",
    round(
        baseline_lane_details["plant_zone_cost_inr"].sum(),
        2
    )
)

print(
    "Total freight wave : ₹",
    round(
        baseline_lane_details["total_lane_cost_inr"].sum(),
        2
    )
)

MODEL A LANE-COST STRUCTURE CHECK

Model A lane-cost columns:
['Plant Code', 'city', 'zone_weight_kg', 'distance_km', 'plant_port_cost_inr', 'plant_zone_cost_inr', 'total_lane_cost_inr', 'is_feasible']

Model A lane-cost summary:
       zone_weight_kg  distance_km  plant_port_cost_inr  plant_zone_cost_inr  total_lane_cost_inr
count      288.000000   288.000000           288.000000           288.000000         2.880000e+02
mean     10173.200029  1318.004861         61309.939730         70732.631184         1.320426e+05
std      12193.488415   731.135368         87073.774081         99040.253210         1.740095e+05
min        297.547486     0.000000          1198.521274             0.000000         1.649008e+03
25%       1554.118818   766.350000         11659.398361          7738.915433         2.064669e+04
50%       5998.706416  1289.200000         30593.318938         26888.423378         6.685217e+04
75%      14341.743358  1822.925000         73179.817739         83934.736737        

In [59]:
print("=" * 100)
print("DISRUPTION ANALYSIS — FORMULATION VALIDATION")
print("=" * 100)

print()
print("Optimized baseline plants:")
print(results[1]["open_plants"])

print()
print("All candidate plants:")
print(plants)

print()
print("=" * 100)
print("SINGLE-PLANT FAILURE OBJECTIVE VALUES")
print("=" * 100)

for r in failure_results:

    print()
    print(f"Failed plant        : {r['failed_plant']}")
    print(f"Status              : {r['status']}")
    print(f"Annual objective    : ₹{r['annual_cost']:,.2f}")
    print(f"Cost increase       : ₹{r['cost_increase']:,.2f}")

    if results[1]["annual_objective"] != 0:
        pct = (
            r["cost_increase"]
            / results[1]["annual_objective"]
            * 100
        )
    else:
        pct = np.nan

    print(f"Cost increase (%)   : {pct:.4f}%")
    print(f"Open plants         : {r['open_plants']}")

    print(
        f"Failed plant open?  : "
        f"{r['failed_plant'] in r['open_plants']}"
    )

    print(
        f"Zones served        : "
        f"{r['assignments']['city'].nunique()}"
    )

    print(
        f"Orders served       : "
        f"{r['assignments']['demand_orders_wave'].sum():,.0f}"
    )


print()
print("=" * 100)
print("FAILURE FORMULATION CHECK")
print("=" * 100)

for r in failure_results:

    failed = r["failed_plant"]

    remaining_candidates = [
        p for p in plants
        if p != failed
    ]

    print()
    print(f"{failed}:")
    print("  Failed plant excluded from model :", failed not in remaining_candidates)
    print("  Remaining candidates available   :", len(remaining_candidates))
    print("  Failed plant reopened            :", failed in r["open_plants"])
    print("  Other plants free to optimize    :", True)

DISRUPTION ANALYSIS — FORMULATION VALIDATION

Optimized baseline plants:
['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

All candidate plants:
['PLANT15', 'PLANT05', 'PLANT02', 'PLANT01', 'PLANT06', 'PLANT10', 'PLANT07', 'PLANT14', 'PLANT16', 'PLANT12', 'PLANT11', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

SINGLE-PLANT FAILURE OBJECTIVE VALUES

Failed plant        : PLANT05
Status              : Optimal
Annual objective    : ₹293,076,561,174.72
Cost increase       : ₹1,026,344,115.91
Cost increase (%)   : 0.3514%
Open plants         : ['PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']
Failed plant open?  : False
Zones served        : 18
Orders served       : 9,215

Failed plant        : PLANT02
Status              : Optimal
Annual objective    : ₹292,197,751,069.28
Cost increase       : ₹147,534,010.47
Cost increase (%)   : 0.0505%
Open plants         : ['PLANT05', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']
Failed plant open?  : False
Zones served

In [60]:
print()
print("=" * 110)
print("FINAL DISRUPTION EVIDENCE TABLE")
print("=" * 110)

disruption_validation = []

baseline_optimized_cost = results[1]["annual_objective"]

for r in failure_results:

    reassigned = (
        r["assignments"]
        .merge(
            results[1]["assignments"][
                ["city", "Plant Code"]
            ],
            on="city",
            how="left",
            suffixes=("_failure", "_baseline")
        )
    )

    reassigned_zones = (
        reassigned[
            reassigned["Plant Code_failure"]
            != reassigned["Plant Code_baseline"]
        ]["city"]
        .nunique()
    )

    disruption_validation.append({
        "failed_plant": r["failed_plant"],
        "status": r["status"],
        "annual_cost": r["annual_cost"],
        "cost_increase": r["cost_increase"],
        "cost_increase_pct":
            r["cost_increase"]
            / baseline_optimized_cost
            * 100,
        "zones_served":
            r["assignments"]["city"].nunique(),
        "orders_served":
            r["assignments"]["demand_orders_wave"].sum(),
        "failed_plant_open":
            r["failed_plant"] in r["open_plants"],
        "reassigned_zones":
            reassigned_zones,
        "open_plants":
            r["open_plants"]
    })

disruption_validation_df = pd.DataFrame(
    disruption_validation
)

print(
    disruption_validation_df[
        [
            "failed_plant",
            "status",
            "annual_cost",
            "cost_increase",
            "cost_increase_pct",
            "zones_served",
            "orders_served",
            "failed_plant_open",
            "reassigned_zones"
        ]
    ]
    .sort_values(
        "cost_increase_pct",
        ascending=False
    )
    .to_string(index=False)
)


FINAL DISRUPTION EVIDENCE TABLE
failed_plant  status  annual_cost  cost_increase  cost_increase_pct  zones_served  orders_served  failed_plant_open  reassigned_zones
     PLANT07 Optimal 3.111332e+11   1.908298e+10           6.534143            18         9215.0              False                13
     PLANT04 Optimal 3.058007e+11   1.375043e+10           4.708243            18         9215.0              False                10
     PLANT13 Optimal 2.943623e+11   2.312118e+09           0.791685            18         9215.0              False                 5
     PLANT05 Optimal 2.930766e+11   1.026344e+09           0.351427            18         9215.0              False                 1
     PLANT03 Optimal 2.925612e+11   5.109560e+08           0.174955            18         9215.0              False                12
     PLANT02 Optimal 2.921978e+11   1.475340e+08           0.050517            18         9215.0              False                 2


In [61]:
zero_zone_rows = baseline_lane_details[
    baseline_lane_details["plant_zone_cost_inr"] == 0
][
    ["Plant Code", "city", "distance_km", "plant_zone_cost_inr"]
]

print(zero_zone_rows.to_string(index=False))

Plant Code    city  distance_km  plant_zone_cost_inr
   PLANT03 Kolkata          0.0                  0.0


In [62]:
print("=" * 100)
print("FINAL MODEL FREEZE — NETWORK OPTIMIZATION")
print("=" * 100)

print()
print("MODEL A BASELINE")
print(f"Annual cost : ₹{results[0]['annual_objective']:,.2f}")
print(f"Plants     : {results[0]['open_plants']}")

print()
print("MODEL A OPTIMIZED")
print(f"Annual cost : ₹{results[1]['annual_objective']:,.2f}")
print(f"Plants     : {results[1]['open_plants']}")

print()
print("MODEL A SAVINGS")

savings = (
    results[0]["annual_objective"]
    - results[1]["annual_objective"]
)

savings_pct = (
    savings
    / results[0]["annual_objective"]
    * 100
)

print(f"Annual savings : ₹{savings:,.2f}")
print(f"Savings %      : {savings_pct:.4f}%")

print()
print("MODEL B BASELINE")
print(f"Annual cost : ₹{results[2]['annual_objective']:,.2f}")

print()
print("MODEL B OPTIMIZED")
print(f"Annual cost : ₹{results[3]['annual_objective']:,.2f}")

print()
print("RESILIENCE")
print("Highest-risk plant : PLANT07")
print("Cost increase      : 6.53%")

print()
print("=" * 100)
print("MODEL STATUS: FROZEN")
print("=" * 100)

FINAL MODEL FREEZE — NETWORK OPTIMIZATION

MODEL A BASELINE
Annual cost : ₹313,429,812,894.86
Plants     : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

MODEL A OPTIMIZED
Annual cost : ₹292,050,217,058.80
Plants     : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

MODEL A SAVINGS
Annual savings : ₹21,379,595,836.05
Savings %      : 6.8212%

MODEL B BASELINE
Annual cost : ₹313,430,868,875.89

MODEL B OPTIMIZED
Annual cost : ₹292,056,637,858.97

RESILIENCE
Highest-risk plant : PLANT07
Cost increase      : 6.53%

MODEL STATUS: FROZEN


In [63]:
import os

print("Current output directory:")
print(processed_path)

print("\nFiles:")
for f in os.listdir(processed_path):
    print(f)

Current output directory:
/content/drive/MyDrive/Distribution_Network_Optimization/Data/Processed

Files:
dataset_profile_summary.csv
warehouse_master_geo.csv
customer_demand_geo.csv
warehouse_master.csv
products_per_plant.csv
warehouse_master_screened.csv
vmi_customer_validation.csv
customer_demand.csv
final_candidate_pool.csv
customer_demand_regional.csv
freight_rates_processed.csv
model_a_lane_costs.csv
model_b_lane_costs.csv
model_a_baseline_assignments.csv
model_a_optimized_assignments.csv
model_b_baseline_assignments.csv
model_b_optimized_assignments.csv
model_a_resilience_analysis.csv
network_optimization_kpis.csv
final_network_results.csv
optimized_capacity_utilization.csv
model_a_vs_model_b.csv


In [64]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/"
    "Distribution_Network_Optimization/Data/Processed"
)

final_outputs = Path(
    "/content/drive/MyDrive/"
    "Distribution_Network_Optimization/Final_Outputs"
)

final_outputs.mkdir(parents=True, exist_ok=True)

final_files = [
    "final_network_results.csv",
    "network_optimization_kpis.csv",
    "optimized_capacity_utilization.csv",
    "model_a_vs_model_b.csv",
    "model_a_resilience_analysis.csv",
    "model_a_baseline_assignments.csv",
    "model_a_optimized_assignments.csv",
    "model_b_baseline_assignments.csv",
    "model_b_optimized_assignments.csv",
    "model_a_lane_costs.csv",
    "model_b_lane_costs.csv",
]

for filename in final_files:
    src = source / filename
    dst = final_outputs / filename

    if src.exists():
        shutil.copy2(src, dst)
        print(f"Copied: {filename}")
    else:
        print(f"NOT FOUND: {filename}")

print()
print("=" * 80)
print("FINAL OUTPUTS FOLDER")
print("=" * 80)

for f in sorted(final_outputs.iterdir()):
    print(f.name)

Copied: final_network_results.csv
Copied: network_optimization_kpis.csv
Copied: optimized_capacity_utilization.csv
Copied: model_a_vs_model_b.csv
Copied: model_a_resilience_analysis.csv
Copied: model_a_baseline_assignments.csv
Copied: model_a_optimized_assignments.csv
Copied: model_b_baseline_assignments.csv
Copied: model_b_optimized_assignments.csv
Copied: model_a_lane_costs.csv
Copied: model_b_lane_costs.csv

FINAL OUTPUTS FOLDER
final_network_results.csv
model_a_baseline_assignments.csv
model_a_lane_costs.csv
model_a_optimized_assignments.csv
model_a_resilience_analysis.csv
model_a_vs_model_b.csv
model_b_baseline_assignments.csv
model_b_lane_costs.csv
model_b_optimized_assignments.csv
network_optimization_kpis.csv
optimized_capacity_utilization.csv


In [65]:
print("=" * 100)
print("FINAL MODEL B VALIDATION")
print("=" * 100)

model_a_baseline = results[0]
model_a_optimized = results[1]
model_b_baseline = results[2]
model_b_optimized = results[3]

# Model B savings
model_b_savings = (
    model_b_baseline["annual_objective"]
    - model_b_optimized["annual_objective"]
)

model_b_savings_pct = (
    model_b_savings
    / model_b_baseline["annual_objective"]
    * 100
)

print()
print("MODEL B BASELINE")
print(f"Annual cost  : ₹{model_b_baseline['annual_objective']:,.2f}")
print(f"Open plants  : {model_b_baseline['open_plants']}")

print()
print("MODEL B OPTIMIZED")
print(f"Annual cost  : ₹{model_b_optimized['annual_objective']:,.2f}")
print(f"Open plants  : {model_b_optimized['open_plants']}")

print()
print("MODEL B SAVINGS")
print(f"Annual savings : ₹{model_b_savings:,.2f}")
print(f"Savings %      : {model_b_savings_pct:.4f}%")

print()
print("=" * 100)
print("MODEL A vs MODEL B — 2-DAY SERVICE PREMIUM")
print("=" * 100)

premium = (
    model_b_optimized["annual_objective"]
    - model_a_optimized["annual_objective"]
)

premium_pct = (
    premium
    / model_a_optimized["annual_objective"]
    * 100
)

print()
print(
    f"Model A optimized : "
    f"₹{model_a_optimized['annual_objective']:,.2f}"
)

print(
    f"Model B optimized : "
    f"₹{model_b_optimized['annual_objective']:,.2f}"
)

print()
print(
    f"Cost of 2-day resilience : "
    f"₹{premium:,.2f}"
)

print(
    f"Premium over Model A     : "
    f"{premium_pct:.4f}%"
)

print()
print("=" * 100)
print("MODEL B — PLANT14 STRUCTURAL CHECK")
print("=" * 100)

plant14_b = model_b_lanes[
    model_b_lanes["Plant Code"] == "PLANT14"
]

print(
    "PLANT14 total lanes        :",
    len(plant14_b)
)

print(
    "PLANT14 feasible 2-day lanes:",
    plant14_b["is_feasible"].sum()
)

print(
    "PLANT14 in Model B optimized:",
    "PLANT14" in model_b_optimized["open_plants"]
)

print()
print("=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)

comparison = pd.DataFrame([
    {
        "Model": "Model A",
        "Scenario": "Baseline",
        "Annual_Cost_INR":
            model_a_baseline["annual_objective"],
        "Open_Plants":
            len(model_a_baseline["open_plants"])
    },
    {
        "Model": "Model A",
        "Scenario": "Optimized",
        "Annual_Cost_INR":
            model_a_optimized["annual_objective"],
        "Open_Plants":
            len(model_a_optimized["open_plants"])
    },
    {
        "Model": "Model B",
        "Scenario": "Baseline",
        "Annual_Cost_INR":
            model_b_baseline["annual_objective"],
        "Open_Plants":
            len(model_b_baseline["open_plants"])
    },
    {
        "Model": "Model B",
        "Scenario": "Optimized",
        "Annual_Cost_INR":
            model_b_optimized["annual_objective"],
        "Open_Plants":
            len(model_b_optimized["open_plants"])
    }
])

print(
    comparison.to_string(index=False)
)

FINAL MODEL B VALIDATION

MODEL B BASELINE
Annual cost  : ₹313,430,868,875.89
Open plants  : ['PLANT16', 'PLANT12', 'PLANT09', 'PLANT03', 'PLANT13', 'PLANT08', 'PLANT04']

MODEL B OPTIMIZED
Annual cost  : ₹292,056,637,858.97
Open plants  : ['PLANT05', 'PLANT02', 'PLANT07', 'PLANT03', 'PLANT13', 'PLANT04']

MODEL B SAVINGS
Annual savings : ₹21,374,231,016.92
Savings %      : 6.8194%

MODEL A vs MODEL B — 2-DAY SERVICE PREMIUM

Model A optimized : ₹292,050,217,058.80
Model B optimized : ₹292,056,637,858.97

Cost of 2-day resilience : ₹6,420,800.17
Premium over Model A     : 0.0022%

MODEL B — PLANT14 STRUCTURAL CHECK
PLANT14 total lanes        : 18
PLANT14 feasible 2-day lanes: 0
PLANT14 in Model B optimized: False

FINAL MODEL COMPARISON
  Model  Scenario  Annual_Cost_INR  Open_Plants
Model A  Baseline     3.134298e+11            7
Model A Optimized     2.920502e+11            6
Model B  Baseline     3.134309e+11            7
Model B Optimized     2.920566e+11            6


In [67]:
from pathlib import Path

model_path = Path(
    "/content/drive/MyDrive/Distribution_Network_Optimization/Model"
)

model_path.mkdir(parents=True, exist_ok=True)

(model_path / "01_Optimization_Model.md").touch()
(model_path / "02_Model_A_vs_Model_B.md").touch()
(model_path / "03_Resilience_Model.md").touch()
(model_path / "04_Assumptions_and_KPIs.md").touch()

print("Model files created:")
for file in sorted(model_path.iterdir()):
    print(file.name)

Model files created:
01_Optimization_Model.md
02_Model_A_vs_Model_B.md
03_Resilience_Model.md
04_Assumptions_and_KPIs.md
